# ###
### B Matrix - differential evolution and nelder mead optimizaiton. We only use the B matrix for urine in this and also knowing the effect of phosphoric acid. Later we recacualte the B matrix for acetate

##

In [ ]:
#!/usr/bin/env python3
"""
DIGESTER v2 — PUMP SHUTOFF CORRECTED
  All Q columns zeroed after 2026-02-15 20:13 UTC
  Training: Feb 15 00:00 -> Feb 17 00:00 (48h)
  Validation: Feb 17 00:00 -> Feb 19 00:00 (48h)
"""
import json, time, sys
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.linalg import expm, solve_discrete_are, inv
try:
    from tqdm import tqdm
except ImportError:
    def tqdm(it, **kw):
        total = kw.get('total', None); desc = kw.get('desc', '')
        lp = -1
        for i, x in enumerate(it):
            if total:
                p = int(100*i/total)
                if p >= lp+10: print(f"  {desc} {p}%", flush=True); lp = p
            yield x
        if total and desc: print(f"  {desc} 100%", flush=True)

np.set_printoptions(precision=6, suppress=True, linewidth=120)

HOME = "./"
POINT_1 = pd.Timestamp("2026-02-15 00:00:00", tz="UTC")
POINT_2 = pd.Timestamp("2026-02-19 00:00:00", tz="UTC")
MIDPOINT = pd.Timestamp("2026-02-17 00:00:00", tz="UTC")
PUMP_OFF = pd.Timestamp("2026-02-15 20:13:00", tz="UTC")
DT = 5.0; nx = 4; MODEL_JSON = "digester_v3_complete.json"

_fc = [0]
def show_fig(fig):
    _fc[0] += 1; fn = f"prof_v2_fig{_fc[0]}.png"
    fig.savefig(fn, dpi=150, facecolor=fig.get_facecolor(), bbox_inches='tight'); plt.close(fig)
    try:
        from IPython.display import display, Image; display(Image(filename=fn))
    except: print(f"  Saved: {fn}")

def r2(a, p):
    ss = np.sum((a-p)**2); st = np.sum((a-a.mean())**2)
    return 1 - ss/st if st > 0 else float('nan')

def rmse(a, p): return np.sqrt(np.mean((a-p)**2))

def load_csv(path):
    df = pd.read_csv(path); df.columns = [c.strip().lower() for c in df.columns]
    df["time"] = pd.to_datetime(df["time"], utc=True)
    df = df.sort_values("time").drop_duplicates(subset="time").set_index("time")
    df["value"] = pd.to_numeric(df["value"], errors="coerce"); return df

def load_all():
    raw = {
        "EC_mS": load_csv(HOME+"Digester Tank 1-cloud_EC_mS.csv"),
        "pH": load_csv(HOME+"Digester Tank 1-cloud_pH.csv"),
        "Temp_C": load_csv(HOME+"Digester Tank 1-cloud_Temp_C.csv"),
        "ORP_mV": load_csv(HOME+"Digester Tank 1-cloud_ORP_mV.csv"),
        "MFC_mV": load_csv(HOME+"Digester Tank 1-cloud_MFC_mV.csv"),
        "Q_urine_mL": load_csv(HOME+"Digester Tank 1-cloud_Q_urine_mL.csv"),
        "Q_acetate_mL": load_csv(HOME+"Digester Tank 1-cloud_Q_spir_mL.csv"),
        "Q_phosphoric_mL": load_csv(HOME+"Digester Tank 1-cloud_Q_kombucha_mL.csv"),
    }
    for nm, d in raw.items(): print(f"    {nm:>20s}: {len(d):>6d} rows")
    return raw

def resample_merge(dfs, freq="5s"):
    all_starts = [df.index.min() for df in dfs.values()]
    sensor_ends = [df.index.max() for n, df in dfs.items() if "Q_" not in n]
    idx = pd.date_range(start=max(all_starts).floor(freq), end=min(sensor_ends).ceil(freq), freq=freq, tz="UTC")
    res = {}; orp_real = None; mfc_real = None
    for name, df in dfs.items():
        if "Q_" in name:
            s = df["value"].resample(freq).sum().reindex(idx).fillna(0.0)
        elif name in ("ORP_mV", "MFC_mV"):
            s = df["value"].resample(freq).mean().reindex(idx)
            real = s.notna()
            if name == "ORP_mV": orp_real = s.index[real]
            else: mfc_real = s.index[real]
            s = s.ffill()
        else:
            s = df["value"].resample(freq).mean().reindex(idx).interpolate(method="time")
        res[name] = s
    merged = pd.DataFrame(res, index=idx).dropna()
    freq_sec = pd.Timedelta(freq).total_seconds()
    merged["Q_acetate_mL"] = (5000.0/12.0) / (3600.0/freq_sec)
    # v2 FIX: zero ALL flow after pump shutoff
    off = merged.index >= PUMP_OFF
    merged.loc[off, "Q_urine_mL"] = 0.0
    merged.loc[off, "Q_acetate_mL"] = 0.0
    merged.loc[off, "Q_phosphoric_mL"] = 0.0
    print(f"  PUMP SHUTOFF FIX: zeroed {off.sum()} rows after {PUMP_OFF}")
    merged["_ORP_real"] = merged.index.isin(orp_real) if orp_real is not None else True
    merged["_MFC_real"] = merged.index.isin(mfc_real) if mfc_real is not None else True
    return merged

def discretize(A, B, G, dt):
    A_d = expm(A * dt)
    M = np.zeros((nx+3, nx+3)); M[:nx,:nx] = A*dt; M[:nx,nx:] = B*dt; B_d = expm(M)[:nx, nx:]
    M2 = np.zeros((nx+1, nx+1)); M2[:nx,:nx] = A*dt; M2[:nx,nx] = G*dt; G_d = expm(M2)[:nx, nx]
    return A_d, B_d, G_d

def design_observer_dare(A_d, H, Q_obs, R_obs):
    P = solve_discrete_are(A_d.T, H.T, Q_obs, R_obs)
    S = H @ P @ H.T + R_obs; L = P @ H.T @ inv(S)
    return L, np.linalg.eigvals(A_d - L @ H), P

BG='#0b1120'; PNL='#111827'; GRD='#1e293b'; TXT='#94a3b8'; TTL='#e2e8f0'; SIM='#22d3ee'
def sty(ax, yl):
    ax.set_facecolor(PNL); ax.set_ylabel(yl, color=TXT, fontsize=10)
    ax.tick_params(colors=TXT, labelsize=9)
    for s in ax.spines.values(): s.set_color(GRD)
    ax.grid(True, alpha=0.2, color=GRD, linewidth=0.5)

def run_observer(A_d, B_d, G_d, L, H, h0, Tr, ec0, ph0, df_data, C_out, biases):
    xr = np.array([ec0, ph0, h0[0], h0[1]])
    oEC = df_data["EC_mS"].values; opH = df_data["pH"].values
    oTp = df_data["Temp_C"].values
    U = df_data[["Q_urine_mL","Q_acetate_mL","Q_phosphoric_mL"]].values
    N = len(oEC)
    orp_real = df_data["_ORP_real"].values if "_ORP_real" in df_data.columns else np.ones(N, dtype=bool)
    mfc_real = df_data["_MFC_real"].values if "_MFC_real" in df_data.columns else np.ones(N, dtype=bool)
    def step(x, temp, u):
        xn = A_d @ (x - xr) + xr + B_d @ u + G_d * (temp - Tr)
        xn[0] = np.clip(xn[0], 0, 30); xn[1] = np.clip(xn[1], 3, 12); return xn
    x_pred = np.zeros((N,nx)); x_corr = np.zeros((N,nx)); innov = np.zeros((N,2))
    x_corr[0] = [oEC[0], opH[0], h0[0], h0[1]]; x_pred[0] = x_corr[0].copy()
    for k in tqdm(range(1,N), desc="Observer", total=N-1):
        xp = step(x_corr[k-1], oTp[k-1], U[k-1]); x_pred[k] = xp
        y = np.array([oEC[k], opH[k]]); inn = y - H @ xp; innov[k] = inn
        xc = xp + L @ inn; xc[0] = np.clip(xc[0], 0, 30); xc[1] = np.clip(xc[1], 3, 12); x_corr[k] = xc
    oORP = df_data["ORP_mV"].values; oMFC = df_data["MFC_mV"].values
    HORIZONS = [5, 10, 30, 60, 120, 300, 600, 1800, 3600]
    horizon = {}
    for h_sec in tqdm(HORIZONS, desc="Horizons"):
        Hs = int(h_sec / DT)
        if Hs >= N: continue
        pE=[]; pP=[]; aE=[]; aP=[]; pO=[]; aO=[]; pM=[]; aM=[]
        for k in range(0, N-Hs, max(1,Hs)):
            target = k + Hs; xf = x_corr[k].copy()
            for s in range(Hs):
                if k+s < N-1: xf = step(xf, oTp[k+s], U[k+s])
            pE.append(xf[0]); pP.append(xf[1]); aE.append(oEC[target]); aP.append(opH[target])
            if orp_real[target]: pO.append(C_out[0]@xf+biases[0]); aO.append(oORP[target])
            if mfc_real[target]: pM.append(C_out[1]@xf+biases[1]); aM.append(oMFC[target])
        pE=np.array(pE);pP=np.array(pP);aE=np.array(aE);aP=np.array(aP)
        pO=np.array(pO);aO=np.array(aO);pM=np.array(pM);aM=np.array(aM)
        horizon[h_sec] = {
            'r2_EC':r2(aE,pE),'r2_pH':r2(aP,pP),
            'r2_ORP':r2(aO,pO) if len(aO)>5 else float('nan'),
            'r2_MFC':r2(aM,pM) if len(aM)>5 else float('nan'),
            'rmse_EC':rmse(aE,pE),'rmse_pH':rmse(aP,pP),
            'rmse_ORP':rmse(aO,pO) if len(aO)>5 else float('nan'),
            'rmse_MFC':rmse(aM,pM) if len(aM)>5 else float('nan'),
            'sec':h_sec,'steps':Hs,'n_orp':len(aO),'n_mfc':len(aM),
        }
    return x_pred, x_corr, innov, horizon, orp_real, mfc_real

def plot_observer_states(df_data, x_pred, x_corr, innov, title_prefix=""):
    t = df_data.index; oEC = df_data["EC_mS"].values; opH = df_data["pH"].values
    fig, axes = plt.subplots(6, 1, figsize=(16, 20), facecolor=BG)
    fig.subplots_adjust(hspace=0.35, left=0.08, right=0.96, top=0.94, bottom=0.03)
    fig.suptitle(f'{title_prefix}Observer (v2 pump-corrected)', fontsize=13, color=TTL, fontweight='bold')
    sn = ['EC (mS)', 'pH', 'h1 (hidden)', 'h2 (hidden)']
    obs_data = [oEC, opH, None, None]; clrs = ['#10b981','#f59e0b','#ef4444','#a78bfa']
    for i in range(4):
        ax = axes[i]; sty(ax, sn[i])
        ax.plot(t, x_corr[:,i], color=SIM, lw=1.5, alpha=0.9, label='Corrected')
        ax.plot(t, x_pred[:,i], color='#f43f5e', lw=0.6, ls='--', alpha=0.4, label='Predicted')
        if obs_data[i] is not None:
            ax.plot(t, obs_data[i], '-', color=clrs[i], lw=0.8, alpha=0.4, label='Sensor')
            ax.text(0.98,0.96,f'R2={r2(obs_data[i],x_corr[:,i]):.4f}',transform=ax.transAxes,ha='right',va='top',fontsize=10,color=SIM,fontfamily='monospace',bbox=dict(boxstyle='round,pad=0.3',facecolor=BG,edgecolor=GRD,alpha=0.9))
        leg = ax.legend(fontsize=7, loc='lower right', framealpha=0.8, facecolor=PNL, edgecolor=GRD)
        for lt in leg.get_texts(): lt.set_color(TTL)
    ax = axes[4]; sty(ax, 'EC innov'); ax.plot(t, innov[:,0], color='#10b981', lw=0.5, alpha=0.7); ax.axhline(0, color='white', ls='--', lw=0.3)
    ax = axes[5]; sty(ax, 'pH innov'); ax.plot(t, innov[:,1], color='#f59e0b', lw=0.5, alpha=0.7); ax.axhline(0, color='white', ls='--', lw=0.3)
    show_fig(fig)

def plot_validation_pred(df_data, x_corr, C_out, biases, A_d, B_d, G_d, Tr, h0, ec0, ph0, horizon_steps=1):
    oEC=df_data["EC_mS"].values; opH=df_data["pH"].values; oORP=df_data["ORP_mV"].values; oMFC=df_data["MFC_mV"].values
    oTp=df_data["Temp_C"].values; U=df_data[["Q_urine_mL","Q_acetate_mL","Q_phosphoric_mL"]].values
    t=df_data.index; N=len(oEC); xr=np.array([ec0,ph0,h0[0],h0[1]])
    def step(x,temp,u):
        xn=A_d@(x-xr)+xr+B_d@u+G_d*(temp-Tr); xn[0]=np.clip(xn[0],0,30); xn[1]=np.clip(xn[1],3,12); return xn
    h_sec=horizon_steps*DT; h_label=f"{h_sec:.0f}s" if h_sec<60 else f"{h_sec/60:.0f}min"
    pa = {s: np.full(N,np.nan) for s in ['EC','pH','ORP','MFC']}
    for k in tqdm(range(N-horizon_steps), desc=f"{h_label}-ahead", total=N-horizon_steps):
        xf = x_corr[k].copy()
        for s in range(horizon_steps):
            if k+s<N-1: xf=step(xf,oTp[k+s],U[k+s])
        pa['EC'][k+horizon_steps]=xf[0]; pa['pH'][k+horizon_steps]=xf[1]
        pa['ORP'][k+horizon_steps]=C_out[0]@xf+biases[0]; pa['MFC'][k+horizon_steps]=C_out[1]@xf+biases[1]
    mask=~np.isnan(pa['EC']); col={'EC':'EC_mS','pH':'pH','ORP':'ORP_mV','MFC':'MFC_mV'}
    r2s={s:r2(df_data[col[s]].values[mask],pa[s][mask]) for s in col}
    fig,axes=plt.subplots(4,1,figsize=(16,14),facecolor=BG)
    fig.subplots_adjust(hspace=0.30,left=0.08,right=0.96,top=0.92,bottom=0.04)
    fig.suptitle(f'Validation {h_label}-ahead (v2 pump-corrected)\npH R2={r2s["pH"]:.4f}  EC R2={r2s["EC"]:.4f}  ORP R2={r2s["ORP"]:.4f}  MFC R2={r2s["MFC"]:.4f}',fontsize=13,color=TTL,fontweight='bold')
    for pi,(yl,obs,pred,clr,r2v) in enumerate([('pH',opH,pa['pH'],'#f59e0b',r2s['pH']),('EC (mS)',oEC,pa['EC'],'#10b981',r2s['EC']),('ORP (mV)',oORP,pa['ORP'],'#ef4444',r2s['ORP']),('MFC (mV)',oMFC,pa['MFC'],'#a78bfa',r2s['MFC'])]):
        ax=axes[pi]; sty(ax,yl); ax.plot(t,obs,'-',color=clr,lw=1.0,alpha=0.6,label='Measured'); ax.plot(t,pred,color=SIM,lw=1.5,alpha=0.9,label=f'{h_label}-ahead')
        ax.text(0.98,0.96,f'R2={r2v:.4f}',transform=ax.transAxes,ha='right',va='top',fontsize=10,color=SIM,fontfamily='monospace',bbox=dict(boxstyle='round,pad=0.3',facecolor=BG,edgecolor=GRD,alpha=0.9))
        leg=ax.legend(fontsize=8,loc='lower right',framealpha=0.8,facecolor=PNL,edgecolor=GRD)
        for lt in leg.get_texts(): lt.set_color(TTL)
    show_fig(fig); return r2s

def plot_horizon(horizon):
    fig,axes=plt.subplots(1,2,figsize=(14,5),facecolor=BG)
    fig.subplots_adjust(wspace=0.25,left=0.08,right=0.96,top=0.88,bottom=0.12)
    fig.suptitle(f'Prediction vs Horizon (v2 pump-corrected)',fontsize=13,color=TTL,fontweight='bold')
    Hs=sorted(horizon.keys()); labels=[f"{s}s" if s<60 else f"{s//60}m" if s<3600 else f"{s//3600}h" for s in Hs]
    ax=axes[0]; sty(ax,'R2')
    for key,clr,lab in [('r2_pH','#f59e0b','pH'),('r2_EC','#10b981','EC'),('r2_ORP','#ef4444','ORP'),('r2_MFC','#a78bfa','MFC')]:
        ax.plot(range(len(Hs)),[horizon[H][key] for H in Hs],'o-',color=clr,lw=2,ms=5,label=lab)
    ax.set_xticks(range(len(Hs))); ax.set_xticklabels(labels,color=TXT); ax.axhline(0,color='white',ls='--',lw=0.5,alpha=0.3)
    ax.set_ylim(bottom=max(-5,min(v for H in Hs for k in ['r2_pH','r2_EC','r2_ORP','r2_MFC'] for v in [horizon[H][k]] if v>-1e6)-0.5),top=1.05)
    leg=ax.legend(fontsize=9,framealpha=0.8,facecolor=PNL,edgecolor=GRD)
    for lt in leg.get_texts(): lt.set_color(TTL)
    ax=axes[1]; sty(ax,'RMSE')
    for key,clr,lab in [('rmse_pH','#f59e0b','pH'),('rmse_EC','#10b981','EC')]:
        ax.plot(range(len(Hs)),[horizon[H][key] for H in Hs],'o-',color=clr,lw=2,ms=5,label=lab)
    ax.set_xticks(range(len(Hs))); ax.set_xticklabels(labels,color=TXT)
    leg=ax.legend(fontsize=9,framealpha=0.8,facecolor=PNL,edgecolor=GRD)
    for lt in leg.get_texts(): lt.set_color(TTL)
    show_fig(fig)

# ===================================================================
#  MAIN
# ===================================================================
if __name__ == "__main__":
    t0 = time.time()
    sn = ["EC","pH","h1","h2"]; inp = ["Q_urine","Q_acetate","Q_phosphoric"]

    print("="*72)
    print(f"  DIGESTER v2 -- PUMP SHUTOFF CORRECTED")
    print(f"  Pump off: {PUMP_OFF} (all Q=0 after this)")
    print("="*72)
    print(f"  Train: {POINT_1} -> {MIDPOINT}")
    print(f"  Valid: {MIDPOINT} -> {POINT_2}")

    print(f"\n  Loading data ..."); raw = load_all()
    df = resample_merge(raw, freq=f"{int(DT)}s")
    print(f"\n  Grid: {len(df)} pts at {DT:.0f}s")
    df_train = df[(df.index>=POINT_1)&(df.index<MIDPOINT)]
    df_val = df[(df.index>=MIDPOINT)&(df.index<=POINT_2)]
    print(f"  Train: {len(df_train)} pts | Valid: {len(df_val)} pts")

    q_cols = ["Q_urine_mL","Q_acetate_mL","Q_phosphoric_mL"]
    before_off = df_train[df_train.index < PUMP_OFF][q_cols].sum().sum()
    after_off = df_train[df_train.index >= PUMP_OFF][q_cols].sum().sum()
    print(f"  Flow before shutoff: {before_off:.2f} mL")
    print(f"  Flow after shutoff:  {after_off:.2f} mL (should be 0)")

    ec0 = 12.1947; ph0 = 8.3422; temp0 = 15.0775
    print(f"\n  Nominal: EC0={ec0:.4f}  pH0={ph0:.4f}  T0={temp0:.2f}")

    # TRAIN
    print(f"\n{'='*72}\n  TRAINING\n{'='*72}")
    CHUNK_H = 3; chunk_size = int(CHUNK_H*3600/DT)
    chunks_df = []
    for s in range(0, len(df_train)-chunk_size+1, chunk_size):
        chunks_df.append(df_train.iloc[s:s+chunk_size])
    rem = len(df_train) - len(chunks_df)*chunk_size
    if rem > chunk_size//2: chunks_df.append(df_train.iloc[len(chunks_df)*chunk_size:])

    CA = []
    for ch in chunks_df:
        CA.append({
            'EC': ch['EC_mS'].values.astype(np.float64),
            'pH': ch['pH'].values.astype(np.float64),
            'U': ch[['Q_urine_mL','Q_acetate_mL','Q_phosphoric_mL']].values.astype(np.float64),
            'T': ch['Temp_C'].values.astype(np.float64),
            'ORP': ch['ORP_mV'].values.astype(np.float64),
            'MFC': ch['MFC_mV'].values.astype(np.float64),
            'N': len(ch),
        })
    print(f"  {len(CA)} chunks x {CHUNK_H}h ({chunk_size} pts)")

    def sim_fast(A, B, G, h_init, cd, dt, ecr, phr, Tr):
        N = cd['N']; x = np.zeros((N,4))
        x[0,0]=cd['EC'][0]; x[0,1]=cd['pH'][0]; x[0,2:]=h_init
        x0 = np.array([ecr, phr, h_init[0], h_init[1]])
        for k in range(N-1):
            x[k+1] = x[k] + dt * (A@(x[k]-x0) + B@cd['U'][k] + G*(cd['T'][k]-Tr))
        return x

    TIME_LIMIT = 25*60
    _ne=[0]; _best=[1e30]; _t=[0.]; _bat=[0]; _bx=[None]
    class _Stop(Exception): pass

    def cost(theta):
        _ne[0]+=1
        if time.time()-_t[0] > TIME_LIMIT: raise _Stop()
        A_=theta[:16].reshape(4,4); B_=theta[16:28].reshape(4,3)
        G_=theta[28:32]; h0_=theta[32:34]; Tr_=theta[34]
        tot = 0.0
        for ci in range(len(CA)):
            cd = CA[ci]; N=cd['N']; x = np.zeros((N,4))
            x[0,0]=cd['EC'][0]; x[0,1]=cd['pH'][0]; x[0,2:]=h0_
            x0 = np.array([ec0, ph0, h0_[0], h0_[1]])
            for k in range(N-1):
                x[k+1] = x[k] + DT * (A_@(x[k]-x0) + B_@cd['U'][k] + G_*(cd['T'][k]-Tr_))
            tot += np.sum((x[:,0]-cd['EC'])**2) + np.sum((x[:,1]-cd['pH'])**2)
        for ev in np.linalg.eigvals(A_):
            if ev.real>0: tot+=1e6*ev.real**2
        if tot < _best[0]*0.9995: _best[0]=tot; _bat[0]=_ne[0]; _bx[0]=theta.copy()
        elif tot < _best[0]: _best[0]=tot; _bx[0]=theta.copy()
        if _ne[0]%500==0:
            print(f"  eval {_ne[0]:>6d}  cost={tot:.2f}  best={_best[0]:.2f}  stall={_ne[0]-_bat[0]}  ({time.time()-_t[0]:.0f}s/{TIME_LIMIT}s)", flush=True)
        return tot

    _t[0]=time.time()
    cost(np.concatenate([np.diag([-0.001,-0.0003,-0.001,-0.001]).ravel(),np.zeros(12),np.zeros(4),np.zeros(2),[temp0]]))
    t1=time.time()-_t[0]; _ne[0]=0
    print(f"  1 eval = {t1:.3f}s -> ~{int(TIME_LIMIT/t1)} evals in 25min")

    try:
        with open(MODEL_JSON) as f: mi=json.load(f)
        theta0=np.concatenate([np.array(mi['A']).ravel(),np.array(mi['B']).ravel(),np.array(mi['G']),np.array(mi['h0']),[float(mi['T_ref'])]])
        print(f"  Initial guess from {MODEL_JSON}")
    except:
        theta0=np.concatenate([np.diag([-0.001,-0.0003,-0.001,-0.001]).ravel(),np.zeros(12),np.zeros(4),np.zeros(2),[temp0]])
        print(f"  Default initial guess")

    print(f"  {len(theta0)} parameters")
    _t[0]=time.time(); ic=cost(theta0); print(f"  Initial cost: {ic:.2f}")
    _ne[0]=0; _best[0]=ic; _bat[0]=0; _bx[0]=theta0.copy(); _t[0]=time.time()

    from scipy.optimize import minimize as sp_minimize
    print(f"\n  Optimizing (25 min limit):")
    try:
        res=sp_minimize(cost,theta0,method='Nelder-Mead',options={'maxiter':50000,'xatol':1e-10,'fatol':1e-10,'adaptive':True})
        bx=res.x; print(f"\n  CONVERGED: cost={res.fun:.4f} evals={_ne[0]} ({time.time()-_t[0]:.0f}s)")
    except _Stop:
        bx=_bx[0]; print(f"\n  EARLY STOP: best={_best[0]:.4f} evals={_ne[0]} ({time.time()-_t[0]:.0f}s)")
    except KeyboardInterrupt:
        bx=_bx[0]; print(f"\n  INTERRUPTED: best={_best[0]:.4f} evals={_ne[0]} ({time.time()-_t[0]:.0f}s)")

    A=bx[:16].reshape(4,4); B=bx[16:28].reshape(4,3); G=bx[28:32]; h0=bx[32:34]; Tr=bx[34]

    # Fit C
    print(f"\n  Fitting C matrix ...")
    orp_rm=df_train["_ORP_real"].values; mfc_rm=df_train["_MFC_real"].values
    axd=[]; aorp=[]; amfc=[]
    for ci,cd in enumerate(CA):
        xs=sim_fast(A,B,G,h0,cd,DT,ec0,ph0,Tr)
        xd=xs.copy(); xd[:,0]-=ec0; xd[:,1]-=ph0
        axd.append(xd); aorp.append(cd['ORP']); amfc.append(cd['MFC'])
    axd=np.vstack(axd); aorp=np.concatenate(aorp); amfc=np.concatenate(amfc)
    om=orp_rm[:len(aorp)]; mm=mfc_rm[:len(amfc)]
    co,_,_,_=np.linalg.lstsq(np.column_stack([axd[om],np.ones(om.sum())]),aorp[om],rcond=None)
    cm,_,_,_=np.linalg.lstsq(np.column_stack([axd[mm],np.ones(mm.sum())]),amfc[mm],rcond=None)
    C_out=np.array([co[:4],cm[:4]]); biases=np.array([co[-1],cm[-1]])
    print(f"  C fit on {om.sum()} ORP, {mm.sum()} MFC real pts")

    aes=[]; aps=[]
    for ci,cd in enumerate(CA):
        xs=sim_fast(A,B,G,h0,cd,DT,ec0,ph0,Tr); aes.append(xs[:,0]); aps.append(xs[:,1])
    aes=np.concatenate(aes); aps=np.concatenate(aps)
    aec=np.concatenate([cd['EC'] for cd in CA]); apc=np.concatenate([cd['pH'] for cd in CA])
    az=(C_out@axd.T).T+biases
    train_m={'r2_EC':r2(aec,aes),'r2_pH':r2(apc,aps),'r2_ORP':r2(aorp[om],az[om,0]),'r2_MFC':r2(amfc[mm],az[mm,1])}

    with open("digester_prof_trained_v2.json","w") as f:
        json.dump({'A':A.tolist(),'B':B.tolist(),'G':G.tolist(),'C':C_out.tolist(),'biases':biases.tolist(),
                   'h0':h0.tolist(),'T_ref':float(Tr),'nominal':{'EC_mS':float(ec0),'pH':float(ph0)},
                   'metrics':train_m,'pump_off':str(PUMP_OFF)},f,indent=2)
    print(f"  Saved: digester_prof_trained_v2.json")

    # PRINT MATRICES
    print(f"\n{'='*72}\n  RESULTS\n{'='*72}")
    print(f"  EC0={ec0:.4f}  pH0={ph0:.4f}  T_ref={Tr:.2f}  h0={h0}")
    print(f"\n  Training metrics:")
    for k,v in train_m.items(): print(f"    {k}: {v:.4f}")

    print(f"\n  A (continuous):")
    print("      "+"".join(f"{n:>10s}" for n in sn))
    for i in range(4): print(f"  {sn[i]:>4s} "+"".join(f"{A[i,j]:>10.6f}" for j in range(4)))
    print(f"\n  Eigenvalues:")
    for i,ev in enumerate(np.linalg.eigvals(A)):
        tc=abs(1/ev.real)/60 if abs(ev.real)>1e-10 else float('inf')
        print(f"    l_{i+1}={ev.real:+.6f} (tau={tc:.1f}min)")

    A_d,B_d,G_d = discretize(A,B,G,DT)
    print(f"\n  A_d = e^(A*{DT:.0f}s):")
    print("      "+"".join(f"{n:>10s}" for n in sn))
    for i in range(4): print(f"  {sn[i]:>4s} "+"".join(f"{A_d[i,j]:>10.6f}" for j in range(4)))
    print(f"\n  Discrete eigenvalues:")
    for i,ev in enumerate(np.linalg.eigvals(A_d)):
        print(f"    z_{i+1}={ev.real:+.8f} |z|={abs(ev):.8f}")

    print(f"\n  B (continuous):")
    print("      "+"".join(f"{n:>14s}" for n in inp))
    for i in range(4): print(f"  {sn[i]:>4s} "+"".join(f"{B[i,j]:>14.6f}" for j in range(3)))
    print(f"\n  B_d (per {DT:.0f}s):")
    print("      "+"".join(f"{n:>14s}" for n in inp))
    for i in range(4): print(f"  {sn[i]:>4s} "+"".join(f"{B_d[i,j]:>14.6f}" for j in range(3)))

    print(f"\n  G -> G_d:")
    for i in range(4): print(f"    {sn[i]:>4s}: G={G[i]:+.8f}  G_d={G_d[i]:+.8f}")
    print(f"\n  C + bias:")
    print("      "+"".join(f"{n:>10s}" for n in sn)+"       bias")
    print(f"  ORP "+"".join(f"{C_out[0,j]:>10.4f}" for j in range(4))+f"  {biases[0]:>10.4f}")
    print(f"  MFC "+"".join(f"{C_out[1,j]:>10.4f}" for j in range(4))+f"  {biases[1]:>10.4f}")

    # OBSERVER DESIGN
    print(f"\n{'='*72}\n  OBSERVER (DARE)\n{'='*72}")
    H = np.zeros((2,4)); H[0,0]=1; H[1,1]=1
    O = np.vstack([H@np.linalg.matrix_power(A_d,i) for i in range(4)])
    print(f"  Observability rank: {np.linalg.matrix_rank(O)}/4")
    Q_obs=np.diag([0.1,0.1,0.001,0.001]); R_obs=np.diag([0.04,0.01])
    try:
        L,obs_eigs,_ = design_observer_dare(A_d, H, Q_obs, R_obs); print(f"  DARE solved")
    except Exception as e:
        from scipy.signal import place_poles as pp
        res=pp(A_d.T,H.T,[0.95,0.93,0.90,0.88]); L=res.gain_matrix.T
        obs_eigs=np.linalg.eigvals(A_d-L@H); print(f"  Pole placement fallback: {e}")

    print(f"\n  L:")
    for i in range(4): print(f"  {sn[i]:>4s}  {L[i,0]:>12.6f}  {L[i,1]:>12.6f}")
    print(f"\n  Observer poles:")
    for i,ev in enumerate(obs_eigs):
        mag=abs(ev); tau=-DT/np.log(mag) if 0<mag<1 else float('inf')
        print(f"    p_{i+1}={ev.real:+.6f} |p|={mag:.6f} tau={tau:.1f}s")

    # OBSERVER ON TRAINING
    print(f"\n{'='*72}\n  OBSERVER ON TRAINING ({len(df_train)} pts)\n{'='*72}")
    xp_tr,xc_tr,inn_tr,hor_tr,orm_tr,mrm_tr = run_observer(A_d,B_d,G_d,L,H,h0,Tr,ec0,ph0,df_train,C_out,biases)
    oEC_tr=df_train["EC_mS"].values; opH_tr=df_train["pH"].values
    oORP_tr=df_train["ORP_mV"].values; oMFC_tr=df_train["MFC_mV"].values
    xd_tr=xc_tr.copy(); xd_tr[:,0]-=ec0; xd_tr[:,1]-=ph0; z_tr=(C_out@xd_tr.T).T+biases
    print(f"  EC  R2={r2(oEC_tr,xc_tr[:,0]):.4f}  RMSE={rmse(oEC_tr,xc_tr[:,0]):.4f}")
    print(f"  pH  R2={r2(opH_tr,xc_tr[:,1]):.4f}  RMSE={rmse(opH_tr,xc_tr[:,1]):.4f}")
    print(f"  ORP R2={r2(oORP_tr[orm_tr],z_tr[orm_tr,0]):.4f}")
    print(f"  MFC R2={r2(oMFC_tr[mrm_tr],z_tr[mrm_tr,1]):.4f}")
    try: plot_observer_states(df_train,xp_tr,xc_tr,inn_tr,"TRAINING -- ")
    except Exception as e: print(f"  Plot error: {e}")

    # OBSERVER ON VALIDATION
    print(f"\n{'='*72}\n  OBSERVER ON VALIDATION ({len(df_val)} pts)\n{'='*72}")
    h_val = xc_tr[-1,2:]
    print(f"  Hidden from training: h1={h_val[0]:.6f} h2={h_val[1]:.6f}")
    xp_v,xc_v,inn_v,hor_v,orm_v,mrm_v = run_observer(A_d,B_d,G_d,L,H,h_val,Tr,ec0,ph0,df_val,C_out,biases)
    oEC_v=df_val["EC_mS"].values; opH_v=df_val["pH"].values
    oORP_v=df_val["ORP_mV"].values; oMFC_v=df_val["MFC_mV"].values
    xd_v=xc_v.copy(); xd_v[:,0]-=ec0; xd_v[:,1]-=ph0; z_v=(C_out@xd_v.T).T+biases
    print(f"  EC  R2={r2(oEC_v,xc_v[:,0]):.4f}  RMSE={rmse(oEC_v,xc_v[:,0]):.4f}")
    print(f"  pH  R2={r2(opH_v,xc_v[:,1]):.4f}  RMSE={rmse(opH_v,xc_v[:,1]):.4f}")
    print(f"  ORP R2={r2(oORP_v[orm_v],z_v[orm_v,0]):.4f}")
    print(f"  MFC R2={r2(oMFC_v[mrm_v],z_v[mrm_v,1]):.4f}")

    print(f"\n  Horizon table:")
    print(f"  {'Horizon':>10s} {'R2_EC':>8s} {'R2_pH':>8s} {'R2_ORP':>8s} {'R2_MFC':>8s} {'RMSE_EC':>8s} {'RMSE_pH':>8s}")
    for hs in sorted(hor_v.keys()):
        h=hor_v[hs]; lab=f"{hs}s" if hs<60 else f"{hs//60}min" if hs<3600 else f"{hs//3600}h"
        print(f"  {lab:>10s} {h['r2_EC']:>8.4f} {h['r2_pH']:>8.4f} {h['r2_ORP']:>8.4f} {h['r2_MFC']:>8.4f} {h['rmse_EC']:>8.4f} {h['rmse_pH']:>8.4f}")

    print(f"\n  COMPARISON:")
    print(f"  {'':>8s} {'Train':>8s} {'Valid':>8s}")
    print(f"  {'EC R2':>8s} {r2(oEC_tr,xc_tr[:,0]):>8.4f} {r2(oEC_v,xc_v[:,0]):>8.4f}")
    print(f"  {'pH R2':>8s} {r2(opH_tr,xc_tr[:,1]):>8.4f} {r2(opH_v,xc_v[:,1]):>8.4f}")
    print(f"  {'ORP R2':>8s} {r2(oORP_tr[orm_tr],z_tr[orm_tr,0]):>8.4f} {r2(oORP_v[orm_v],z_v[orm_v,0]):>8.4f}")
    print(f"  {'MFC R2':>8s} {r2(oMFC_tr[mrm_tr],z_tr[mrm_tr,1]):>8.4f} {r2(oMFC_v[mrm_v],z_v[mrm_v,1]):>8.4f}")

    # PLOTS
    try: plot_observer_states(df_val,xp_v,xc_v,inn_v,"VALIDATION -- ")
    except Exception as e: print(f"  Error: {e}")
    try: plot_validation_pred(df_val,xc_v,C_out,biases,A_d,B_d,G_d,Tr,h_val,ec0,ph0,horizon_steps=1)
    except Exception as e: print(f"  Error: {e}")
    try: plot_validation_pred(df_val,xc_v,C_out,biases,A_d,B_d,G_d,Tr,h_val,ec0,ph0,horizon_steps=12)
    except Exception as e: print(f"  Error: {e}")
    try: plot_horizon(hor_v)
    except Exception as e: print(f"  Error: {e}")

    # SAVE
    res_json = {
        'dt':DT,'A':A.tolist(),'B':B.tolist(),'G':G.tolist(),
        'A_d':A_d.tolist(),'B_d':B_d.tolist(),'G_d':G_d.tolist(),
        'C':C_out.tolist(),'biases':biases.tolist(),'L':L.tolist(),'H':H.tolist(),
        'h0':h0.tolist(),'T_ref':float(Tr),'nominal':{'EC_mS':float(ec0),'pH':float(ph0)},
        'train_metrics':train_m,'validation_horizon':{str(k):v for k,v in hor_v.items()},
        'pump_off':str(PUMP_OFF),
        'version': 'v2 pump shutoff corrected',
    }
    with open("digester_prof_final_v2.json","w") as f: json.dump(res_json,f,indent=2)
    print(f"\n  Saved: digester_prof_final_v2.json")
    print(f"  Total time: {time.time()-t0:.0f}s")
    print(f"\nDONE!")

========================================================================
  DIGESTER v2 -- PUMP SHUTOFF CORRECTED
  Pump off: 2026-02-15 20:13:00+00:00 (all Q=0 after this)
========================================================================
  Train: 2026-02-15 00:00:00+00:00 -> 2026-02-17 00:00:00+00:00
  Valid: 2026-02-17 00:00:00+00:00 -> 2026-02-19 00:00:00+00:00

  Loading data ...
                   EC_mS: 168741 rows
                      pH: 168265 rows
                  Temp_C:  39611 rows
                  ORP_mV: 118817 rows
                  MFC_mV: 111827 rows
              Q_urine_mL:  10917 rows
            Q_acetate_mL:   7873 rows
         Q_phosphoric_mL:  12852 rows
  PUMP SHUTOFF FIX: zeroed 144789 rows after 2026-02-15 20:13:00+00:00

  Grid: 164476 pts at 5s
  Train: 34560 pts | Valid: 34561 pts
  Flow before shutoff: 18325.69 mL
  Flow after shutoff:  0.00 mL (should be 0)

  Nominal: EC0=12.1947  pH0=8.3422  T0=15.08

========================================================================
  TRAINING
========================================================================
  16 chunks x 3h (2160 pts)
  1 eval = 0.577s -> ~2597 evals in 25min
  Initial guess from digester_v3_complete.json
  35 parameters
  Initial cost: 917707.57

  Optimizing (25 min limit):
  eval    500  cost=313626.62  best=313626.62  stall=0  (291s/1500s)
  eval   1000  cost=97609.00  best=97495.33  stall=1  (581s/1500s)
  eval   1500  cost=47765.96  best=47765.96  stall=0  (866s/1500s)
  eval   2000  cost=35811.44  best=35384.12  stall=4  (1149s/1500s)
  eval   2500  cost=18640.23  best=18324.04  stall=44  (1433s/1500s)

  EARLY STOP: best=17486.1693 evals=2620 (1500s)

  Fitting C matrix ...
  C fit on 21977 ORP, 20284 MFC real pts
  Saved: digester_prof_trained_v2.json

========================================================================
  RESULTS
========================================================================
  EC0=12.1947  pH0=8.3422  T_ref=14.07  h0=[-0.000034 -0.000004]

  Training metrics:
    r2_EC: 0.9207
    r2_pH: 0.7828
    r2_ORP: 0.5355
    r2_MFC: 0.1912

  A (continuous):
              EC        pH        h1        h2
    EC  -0.003140  0.004951  0.000151  0.000125
    pH   0.000225 -0.000393  0.000009 -0.000028
    h1   0.000088 -0.000152 -0.001050 -0.000058
    h2   0.000114 -0.000114  0.000007 -0.001295

  Eigenvalues:
    l_1=-0.003512 (tau=4.7min)
    l_2=-0.000035 (tau=474.7min)
    l_3=-0.001283 (tau=13.0min)
    l_4=-0.001047 (tau=15.9min)

  A_d = e^(A*5s):
              EC        pH        h1        h2
    EC   0.984438  0.024537  0.000750  0.000616
    pH   0.001117  0.998050  0.000045 -0.000137
    h1   0.000436 -0.000752  0.994766 -0.000286
    h2   0.000564 -0.000562  0.000034  0.993548

  Discrete eigenvalues:
    z_1=+0.98259082 |z|=0.98259082
    z_2=+0.99982447 |z|=0.99982447
    z_3=+0.99360688 |z|=0.99360688
    z_4=+0.99477886 |z|=0.99477886

  B (continuous):
             Q_urine     Q_acetate  Q_phosphoric
    EC       0.003243     -0.008060      0.000579
    pH       0.000206      0.000563     -0.000205
    h1       0.000007      0.000012      0.000016
    h2       0.000043      0.000081     -0.000008

  B_d (per 5s):
             Q_urine     Q_acetate  Q_phosphoric
    EC       0.016100     -0.039953      0.002862
    pH       0.001037      0.002790     -0.001021
    h1       0.000036      0.000051      0.000083
    h2       0.000216      0.000393     -0.000040

  G -> G_d:
      EC: G=-0.00106537  G_d=-0.00528187
      pH: G=+0.00005066  G_d=+0.00025006
      h1: G=+0.00012909  G_d=+0.00064246
      h2: G=+0.00004523  G_d=+0.00022384

  C + bias:
              EC        pH        h1        h2       bias
  ORP   -19.3466  -30.8660 -109.5466  -17.5340   -130.5048
  MFC     1.9122   15.0640  -11.9504 -139.5537    366.7989

========================================================================
  OBSERVER (DARE)
========================================================================
  Observability rank: 4/4
  DARE solved

  L:
    EC      0.764191      0.000521
    pH      0.000130      0.916054
    h1      0.000639     -0.000022
    h2      0.000485     -0.000145

  Observer poles:
    p_1=+0.220418 |p|=0.220418 tau=3.3s
    p_2=+0.081825 |p|=0.081825 tau=2.0s
    p_3=+0.994758 |p|=0.994758 tau=951.3s
    p_4=+0.993556 |p|=0.993556 tau=773.4s

========================================================================
  OBSERVER ON TRAINING (34560 pts)
========================================================================
Observer: 100%|██████████| 34559/34559 [00:05<00:00, 6646.29it/s]
Horizons: 100%|██████████| 9/9 [00:22<00:00,  2.55s/it]
  EC  R2=0.9987  RMSE=0.0743
  pH  R2=0.9995  RMSE=0.0187
  ORP R2=0.5442
  MFC R2=-0.1167
Observer: 100%|██████████| 34560/34560 [00:05<00:00, 6476.24it/s]
Horizons: 100%|██████████| 9/9 [00:23<00:00,  2.61s/it]
  EC  R2=0.9478  RMSE=0.0828
  pH  R2=0.9936  RMSE=0.0177
  ORP R2=-0.9062
  MFC R2=-0.1350

  Horizon table:
     Horizon    R2_EC    R2_pH   R2_ORP   R2_MFC  RMSE_EC  RMSE_pH
          5s   0.0635   0.1004 -167.1191 -61.6240   0.3508   0.2105
         10s   0.7962   0.9024 -168.9230 -65.4035   0.1563   0.0716
         30s   0.6629   0.8209 -171.2015 -63.7252   0.2015   0.0969
        1min   0.4888   0.7420 -178.9103 -66.5145   0.2479   0.1162
        2min   0.1429   0.6279 -179.2214 -55.4913   0.3284   0.1374
        5min  -1.0161   0.5689 -165.6790 -108.9265   0.4977   0.1494
       10min  -2.2457   0.5443 -166.4925 -91.9023   0.6296   0.1562
       30min  -3.2989   0.3942 -192.5768 -53.2876   0.6708   0.1876
          1h  -2.5696  -0.1777 -277.9251 -156.4771   0.5652   0.2672

  COMPARISON:
              Train    Valid
     EC R2   0.9987   0.9478
     pH R2   0.9995   0.9936
    ORP R2   0.5442  -0.9062
    MFC R2  -0.1167  -0.1350

5s-ahead: 100%|██████████| 34560/34560 [00:03<00:00, 11157.50it/s]

1min-ahead: 100%|██████████| 34549/34549 [00:31<00:00, 1107.11it/s]



  Saved: digester_prof_final_v2.json
  Total time: 1629s

DONE!

################################
### CALCULATE B MATRIX for acetate only - Nelder-Mead  
################################

In [7]:
#!/usr/bin/env python3
"""
retrain_v5.py
=============
Retrains the full digester model on current March 2026 operating conditions.

STEP 1 — Retrain A, B, G, h0, T_ref together (all 35 parameters)
    Training window : 2026-03-15 21:52:00 -> 2026-03-16 00:09:36 UTC
    (~2.3 hours covering all 3 sysid phases at current operating point)
    Nominal point computed from first 10 min of training window
    Warm starts from v5 -> v4 -> v2 if JSON exists

STEP 2 — Refit C matrix (ORP and MFC output mapping)
    Least-squares on the full training window

STEP 3 — Redesign observer (DARE)
    Same Q_obs, R_obs as v4

STEP 4 — Save digester_prof_final_v5.json
    Run script multiple times for additional 25-min optimisation passes

Run on SCC:
    module load python3/3.10
    pip install --user numpy pandas scipy
    python retrain_v5.py

File locations (HOME = "./"):
    digester_prof_final_v2.json       <- warm start fallback
    Digester Tank 1-cloud_EC_mS.csv
    Digester Tank 1-cloud_pH.csv
    Digester Tank 1-cloud_Temp_C.csv
    Digester Tank 1-cloud_ORP_mV.csv
    Digester Tank 1-cloud_MFC_mV.csv
    Digester Tank 1-cloud_Q_urine_mL.csv
    Digester Tank 1-cloud_Q_spir_mL.csv
    Digester Tank 1-cloud_Q_kombucha_mL.csv
"""

import json, time, sys, os
import numpy as np
import pandas as pd
from scipy.linalg import expm, solve_discrete_are, inv
from scipy.optimize import minimize as sp_minimize

np.set_printoptions(precision=6, suppress=True, linewidth=120)

# ── training window ────────────────────────────────────────────────────────────
TRAIN_START = pd.Timestamp("2026-03-15 21:52:00", tz="UTC")
TRAIN_END   = pd.Timestamp("2026-03-16 00:09:36", tz="UTC")

# ── settings ───────────────────────────────────────────────────────────────────
HOME       = "./"
DT         = 5.0
nx         = 4
TIME_LIMIT = 25 * 60   # 25 min per run — re-run for more passes

CSV_FILES = {
    "EC_mS":           HOME + "Digester Tank 1-cloud_EC_mS.csv",
    "pH":              HOME + "Digester Tank 1-cloud_pH.csv",
    "Temp_C":          HOME + "Digester Tank 1-cloud_Temp_C.csv",
    "ORP_mV":          HOME + "Digester Tank 1-cloud_ORP_mV.csv",
    "MFC_mV":          HOME + "Digester Tank 1-cloud_MFC_mV.csv",
    "Q_urine_mL":      HOME + "Digester Tank 1-cloud_Q_urine_mL.csv",
    "Q_acetate_mL":    HOME + "Digester Tank 1-cloud_Q_spir_mL.csv",
    "Q_phosphoric_mL": HOME + "Digester Tank 1-cloud_Q_kombucha_mL.csv",
}

# warm start priority
WARM_JSONS = [
    HOME + "digester_prof_final_v5.json",
    HOME + "digester_prof_final_v4.json",
    HOME + "digester_prof_final_v2.json",
    HOME + "digester_prof_trained_v2.json",
]


# ══════════════════════════════════════════════════════════════════════════════
#  HELPERS
# ══════════════════════════════════════════════════════════════════════════════

def r2(a, p):
    ss = np.sum((a - p) ** 2); st = np.sum((a - a.mean()) ** 2)
    return 1 - ss / st if st > 0 else float("nan")

def rmse(a, p):
    return np.sqrt(np.mean((a - p) ** 2))

def discretize(A, B, G, dt):
    A_d = expm(A * dt)
    M_  = np.zeros((nx + 3, nx + 3))
    M_[:nx, :nx] = A * dt; M_[:nx, nx:] = B * dt
    B_d = expm(M_)[:nx, nx:]
    M2  = np.zeros((nx + 1, nx + 1))
    M2[:nx, :nx] = A * dt; M2[:nx, nx] = G * dt
    G_d = expm(M2)[:nx, nx]
    return A_d, B_d, G_d

def design_observer_dare(A_d, H, Q_obs, R_obs):
    P = solve_discrete_are(A_d.T, H.T, Q_obs, R_obs)
    S = H @ P @ H.T + R_obs
    L = P @ H.T @ inv(S)
    return L, np.linalg.eigvals(A_d - L @ H), P


# ══════════════════════════════════════════════════════════════════════════════
#  DATA LOADING
# ══════════════════════════════════════════════════════════════════════════════

def load_csv(path):
    df = pd.read_csv(path)
    df.columns = [c.strip().lower() for c in df.columns]
    df["time"] = pd.to_datetime(df["time"], utc=True)
    df = df.sort_values("time").drop_duplicates(subset="time").set_index("time")
    df["value"] = pd.to_numeric(df["value"], errors="coerce")
    return df

def load_all():
    raw = {}
    print("  Loading CSVs:")
    for name, path in CSV_FILES.items():
        if not os.path.exists(path):
            print(f"    MISSING: {path}"); sys.exit(1)
        raw[name] = load_csv(path)
        print(f"    {name:>22s}: {len(raw[name]):>6d} rows")
    return raw

def resample_merge(dfs, freq="5s"):
    all_starts  = [df.index.min() for df in dfs.values()]
    sensor_ends = [df.index.max() for n, df in dfs.items() if "Q_" not in n]
    idx = pd.date_range(
        start=max(all_starts).floor(freq),
        end=min(sensor_ends).ceil(freq),
        freq=freq, tz="UTC"
    )
    res = {}; orp_real = mfc_real = None
    for name, df in dfs.items():
        if "Q_" in name:
            s = df["value"].resample(freq).sum().reindex(idx).fillna(0.0)
        elif name in ("ORP_mV", "MFC_mV"):
            s = df["value"].resample(freq).mean().reindex(idx)
            real = s.notna()
            if name == "ORP_mV":  orp_real = s.index[real]
            else:                 mfc_real = s.index[real]
            s = s.ffill()
        else:
            s = df["value"].resample(freq).mean().reindex(idx).interpolate(method="time")
        res[name] = s
    merged = pd.DataFrame(res, index=idx).dropna()
    merged["_ORP_real"] = merged.index.isin(orp_real) if orp_real is not None else True
    merged["_MFC_real"] = merged.index.isin(mfc_real) if mfc_real is not None else True
    return merged


# ══════════════════════════════════════════════════════════════════════════════
#  STEP 1 — RETRAIN A, B, G, h0, T_ref  (35 parameters, identical to v2 Cell 4)
# ══════════════════════════════════════════════════════════════════════════════

def retrain_AB(df_train, ec0, ph0):
    _EC  = df_train["EC_mS"].values.copy()
    _pH  = df_train["pH"].values.copy()
    _U   = df_train[["Q_urine_mL", "Q_acetate_mL", "Q_phosphoric_mL"]].values.copy()
    _T   = df_train["Temp_C"].values.copy()
    _N   = len(_EC)
    _x   = np.zeros((_N, nx), dtype=np.float64)

    print(f"  Training arrays: {_N} pts  ({_N * DT / 3600:.2f} h)  "
          f"buffer {_x.nbytes/1e6:.1f} MB")

    def sim_fast(A_, B_, G_, h0_, Tr_):
        x0 = np.array([ec0, ph0, h0_[0], h0_[1]])
        _x[0, 0] = _EC[0]; _x[0, 1] = _pH[0]
        _x[0, 2] = h0_[0]; _x[0, 3] = h0_[1]
        for k in range(_N - 1):
            _x[k+1] = (_x[k] + DT * (A_ @ (_x[k] - x0) +
                                      B_ @ _U[k] +
                                      G_ * (_T[k] - Tr_)))
        return (np.sum((_x[:, 0] - _EC) ** 2) +
                np.sum((_x[:, 1] - _pH) ** 2))

    _ne  = [0]; _best = [1e30]; _t0 = [0.0]
    _bat = [0]; _bx   = [None]

    class _Stop(Exception):
        pass

    def cost(theta):
        _ne[0] += 1
        if time.time() - _t0[0] > TIME_LIMIT:
            raise _Stop()
        A_  = theta[:16].reshape(4, 4)
        B_  = theta[16:28].reshape(4, 3)
        G_  = theta[28:32]
        h0_ = theta[32:34]
        Tr_ = theta[34]
        # ── physical constraints — hard penalties ──────────────────
        # T_ref must be between 10 and 40 C
        if not (10.0 < Tr_ < 40.0):
            return _best[0] + 1e8
        # h0 must be small (hidden states are deviations, not absolutes)
        if np.any(np.abs(h0_) > 1.0):
            return _best[0] + 1e8
        # acetate must lower pH (B[1,1] < 0)
        if B_[1, 1] > 0:
            return _best[0] + 1e8
        # phosphoric must lower pH (B[1,2] < 0)
        if B_[1, 2] > 0:
            return _best[0] + 1e8
        # cheap stability pre-check
        evs    = np.linalg.eigvals(A_)
        max_ev = max(ev.real for ev in evs)
        if max_ev > 0.01:
            return _best[0] + 1e6 * max_ev ** 2
        tot = sim_fast(A_, B_, G_, h0_, Tr_)
        for ev in evs:
            if ev.real > 0:
                tot += 1e6 * ev.real ** 2
        if tot < _best[0] * 0.9995:
            _best[0] = tot; _bat[0] = _ne[0]; _bx[0] = theta.copy()
        elif tot < _best[0]:
            _best[0] = tot; _bx[0] = theta.copy()
        if _ne[0] % 500 == 0:
            print(f"  eval {_ne[0]:>6d}  cost={tot:.4f}  best={_best[0]:.4f}  "
                  f"stall={_ne[0]-_bat[0]}  ({time.time()-_t0[0]:.0f}s/{TIME_LIMIT}s)",
                  flush=True)
        return tot

    # ── warm start ────────────────────────────────────────────────────────
    theta0 = None
    for wj in WARM_JSONS:
        if os.path.exists(wj):
            with open(wj) as f:
                wm = json.load(f)
            A_w  = np.array(wm["A"]).ravel()
            B_w  = np.array(wm["B"]).ravel()
            G_w  = np.array(wm["G"])
            h0_w = np.array(wm["h0"])
            Tr_w = float(wm["T_ref"])
            # sanitise warm start values
            if not (10.0 < Tr_w < 40.0):
                print(f"  NOTE: T_ref={Tr_w:.2f} in {wj} is unphysical — resetting to {_T.mean():.1f}")
                Tr_w = float(_T.mean())
            if np.any(np.abs(h0_w) > 1.0):
                print(f"  NOTE: h0={h0_w} in {wj} has diverged — resetting to [0,0]")
                h0_w = np.array([0.0, 0.0])
            # fix B sign violations in warm start
            B_w_mat = B_w.reshape(4, 3)
            if B_w_mat[1, 1] > 0: B_w_mat[1, 1] = -abs(B_w_mat[1, 1])
            if B_w_mat[1, 2] > 0: B_w_mat[1, 2] = -abs(B_w_mat[1, 2])
            B_w = B_w_mat.ravel()
            theta0 = np.concatenate([A_w, B_w, G_w, h0_w, [Tr_w]])
            print(f"  Warm start from {wj}  (T_ref={Tr_w:.1f}  h0={h0_w})")
            break
    if theta0 is None:
        T_guess = _T.mean()
        theta0 = np.concatenate([
            np.diag([-0.001, -0.0003, -0.001, -0.001]).ravel(),
            np.zeros(12),   # B
            np.zeros(4),    # G
            np.zeros(2),    # h0
            [T_guess]       # T_ref
        ])
        print("  Cold start (default diagonal A, zero B)")

    print(f"  35 parameters (A:16, B:12, G:4, h0:2, T_ref:1)")
    _t0[0] = time.time()
    ic = cost(theta0)
    print(f"  Initial cost: {ic:.4f}")
    _ne[0] = 0; _best[0] = ic; _bat[0] = 0
    _bx[0] = theta0.copy(); _t0[0] = time.time()

    print(f"  Optimising for {TIME_LIMIT//60:.0f} min ...")
    try:
        res = sp_minimize(cost, theta0, method="Nelder-Mead",
                          options={"maxiter": 100000, "xatol": 1e-10,
                                   "fatol": 1e-10, "adaptive": True})
        bx = res.x
        print(f"\n  CONVERGED: cost={res.fun:.4f}  evals={_ne[0]}  "
              f"({time.time()-_t0[0]:.0f}s)")
    except _Stop:
        bx = _bx[0]
        print(f"\n  TIME LIMIT: best={_best[0]:.4f}  evals={_ne[0]}")
    except KeyboardInterrupt:
        bx = _bx[0]
        print(f"\n  INTERRUPTED: best={_best[0]:.4f}  evals={_ne[0]}")

    A  = bx[:16].reshape(4, 4)
    B  = bx[16:28].reshape(4, 3)
    G  = bx[28:32]
    h0 = bx[32:34]
    Tr = bx[34]

    # final quality check
    x0 = np.array([ec0, ph0, h0[0], h0[1]])
    _x[0, 0] = _EC[0]; _x[0, 1] = _pH[0]
    _x[0, 2] = h0[0];  _x[0, 3] = h0[1]
    for k in range(_N - 1):
        _x[k+1] = _x[k] + DT * (A @ (_x[k] - x0) + B @ _U[k] + G * (_T[k] - Tr))

    print(f"\n  Fit quality:")
    print(f"    EC  R2={r2(_EC, _x[:,0]):.4f}  RMSE={rmse(_EC, _x[:,0]):.4f}")
    print(f"    pH  R2={r2(_pH, _x[:,1]):.4f}  RMSE={rmse(_pH, _x[:,1]):.4f}")

    print(f"\n  Eigenvalues of A:")
    for i, ev in enumerate(np.linalg.eigvals(A)):
        tc = abs(1/ev.real)/60 if abs(ev.real) > 1e-10 else float("inf")
        print(f"    l_{i+1}={ev.real:+.6f}  tau={tc:.1f} min  stable={ev.real<0}")
    print(f"  h0={h0}  T_ref={Tr:.2f}")

    return A, B, G, h0, Tr


# ══════════════════════════════════════════════════════════════════════════════
#  STEP 2 — REFIT C MATRIX
# ══════════════════════════════════════════════════════════════════════════════

def refit_C(df_train, A, B, G, h0, Tr, ec0, ph0):
    N   = len(df_train)
    EC  = df_train["EC_mS"].values
    pH  = df_train["pH"].values
    U   = df_train[["Q_urine_mL","Q_acetate_mL","Q_phosphoric_mL"]].values
    T   = df_train["Temp_C"].values
    ORP = df_train["ORP_mV"].values
    MFC = df_train["MFC_mV"].values
    orm = df_train["_ORP_real"].values
    mrm = df_train["_MFC_real"].values

    x = np.zeros((N, nx))
    x[0] = [EC[0], pH[0], h0[0], h0[1]]
    x0   = np.array([ec0, ph0, h0[0], h0[1]])
    for k in range(N - 1):
        x[k+1] = x[k] + DT * (A @ (x[k] - x0) + B @ U[k] + G * (T[k] - Tr))

    xd = x.copy(); xd[:, 0] -= ec0; xd[:, 1] -= ph0

    co, _, _, _ = np.linalg.lstsq(
        np.column_stack([xd[orm], np.ones(orm.sum())]), ORP[orm], rcond=None)
    cm, _, _, _ = np.linalg.lstsq(
        np.column_stack([xd[mrm], np.ones(mrm.sum())]), MFC[mrm], rcond=None)

    C_out  = np.array([co[:4], cm[:4]])
    biases = np.array([co[-1], cm[-1]])
    print(f"  C fit on {orm.sum()} ORP pts, {mrm.sum()} MFC pts")
    return C_out, biases


# ══════════════════════════════════════════════════════════════════════════════
#  MAIN
# ══════════════════════════════════════════════════════════════════════════════

def main():
    t_wall = time.time()
    sn  = ["EC", "pH", "h1", "h2"]
    inp = ["Q_urine", "Q_acetate", "Q_phosphoric"]

    print("=" * 72)
    print("  DIGESTER v5 — RETRAIN A + B + G ON MARCH 2026 DATA")
    print(f"  Train : {TRAIN_START}  ->  {TRAIN_END}")
    print("=" * 72)

    # ── load & resample ───────────────────────────────────────────────────
    print("\n  Loading data ...")
    raw    = load_all()
    merged = resample_merge(raw, freq=f"{int(DT)}s")
    print(f"  Full grid: {len(merged)} pts")

    df_train = merged[(merged.index >= TRAIN_START) &
                      (merged.index <= TRAIN_END)].copy()

    if len(df_train) < 100:
        print(f"  ERROR: only {len(df_train)} pts in training window")
        sys.exit(1)

    print(f"  Train: {len(df_train)} pts  ({len(df_train)*DT/3600:.2f} h)")

    # nominal from first 10 min (median pH avoids MFC gate spikes)
    first_10 = df_train.iloc[:120]
    ec0   = float(first_10["EC_mS"].mean())
    ph0   = float(first_10["pH"].median())
    temp0 = float(first_10["Temp_C"].mean())
    print(f"  Nominal: EC0={ec0:.4f}  pH0={ph0:.4f}  T0={temp0:.2f}")

    # ── STEP 1: retrain A + B ─────────────────────────────────────────────
    print("\n" + "=" * 72)
    print("  STEP 1 — RETRAIN A, B, G, h0, T_ref  (35 parameters)")
    print("=" * 72)
    A, B, G, h0, Tr = retrain_AB(df_train, ec0, ph0)

    # ── STEP 2: refit C ───────────────────────────────────────────────────
    print("\n" + "=" * 72)
    print("  STEP 2 — REFIT C MATRIX")
    print("=" * 72)
    C_out, biases = refit_C(df_train, A, B, G, h0, Tr, ec0, ph0)
    print(f"\n  C + bias:")
    print("  " + "".join(f"{n:>10s}" for n in sn) + "       bias")
    print(f"  ORP " + "".join(f"{C_out[0,j]:>10.4f}" for j in range(4)) +
          f"  {biases[0]:>10.4f}")
    print(f"  MFC " + "".join(f"{C_out[1,j]:>10.4f}" for j in range(4)) +
          f"  {biases[1]:>10.4f}")

    # ── STEP 3: discretise + observer ────────────────────────────────────
    print("\n" + "=" * 72)
    print("  STEP 3 — DISCRETISE + OBSERVER (DARE)")
    print("=" * 72)
    A_d, B_d, G_d = discretize(A, B, G, DT)

    H     = np.zeros((2, 4)); H[0, 0] = 1; H[1, 1] = 1
    O_mat = np.vstack([H @ np.linalg.matrix_power(A_d, i) for i in range(4)])
    print(f"  Observability rank: {np.linalg.matrix_rank(O_mat)}/4")

    Q_obs = np.diag([0.1, 0.1, 0.001, 0.001])
    R_obs = np.diag([0.04, 0.01])
    try:
        L, obs_eigs, _ = design_observer_dare(A_d, H, Q_obs, R_obs)
        print("  DARE solved")
    except Exception as e:
        print(f"  DARE failed ({e}) — falling back to pole placement")
        from scipy.signal import place_poles
        res_pp = place_poles(A_d.T, H.T, [0.95, 0.93, 0.90, 0.88])
        L = res_pp.gain_matrix.T
        obs_eigs = np.linalg.eigvals(A_d - L @ H)

    print(f"\n  L:")
    for i in range(4):
        print(f"    {sn[i]:>4s}  {L[i,0]:>12.6f}  {L[i,1]:>12.6f}")
    print(f"\n  Observer poles:")
    for i, ev in enumerate(obs_eigs):
        mag = abs(ev)
        tau = -DT / np.log(mag) if 0 < mag < 1 else float("inf")
        print(f"    p_{i+1}={ev.real:+.6f}  |p|={mag:.6f}  tau={tau:.1f}s")

    # ── print full results ────────────────────────────────────────────────
    print("\n" + "=" * 72)
    print("  RESULTS")
    print("=" * 72)

    print(f"\n  Nominal: EC0={ec0:.4f}  pH0={ph0:.4f}  T_ref={Tr:.2f}  h0={h0}")

    print(f"\n  A (continuous):")
    print("  " + "".join(f"{n:>10s}" for n in sn))
    for i in range(4):
        print(f"  {sn[i]:>4s}" + "".join(f"{A[i,j]:>10.6f}" for j in range(4)))

    print(f"\n  B (continuous):")
    print("  " + "".join(f"{n:>16s}" for n in inp))
    for i in range(4):
        print(f"  {sn[i]:>4s}" + "".join(f"{B[i,j]:>16.6f}" for j in range(3)))

    print(f"\n  B_d (discrete, per {DT:.0f} s):")
    print("  " + "".join(f"{n:>16s}" for n in inp))
    for i in range(4):
        print(f"  {sn[i]:>4s}" + "".join(f"{B_d[i,j]:>16.6f}" for j in range(3)))

    print(f"\n  A_d = e^(A*{DT:.0f}s):")
    print("  " + "".join(f"{n:>10s}" for n in sn))
    for i in range(4):
        print(f"  {sn[i]:>4s}" + "".join(f"{A_d[i,j]:>10.6f}" for j in range(4)))

    # ── save ──────────────────────────────────────────────────────────────
    out = {
        "version":  "v5 — A+B+G retrained on Mar 2026 sysid data",
        "dt":       DT,
        "A":        A.tolist(),
        "B":        B.tolist(),
        "G":        G.tolist(),
        "A_d":      A_d.tolist(),
        "B_d":      B_d.tolist(),
        "G_d":      G_d.tolist(),
        "C":        C_out.tolist(),
        "biases":   biases.tolist(),
        "L":        L.tolist(),
        "H":        H.tolist(),
        "h0":       h0.tolist(),
        "T_ref":    float(Tr),
        "nominal":  {"EC_mS": float(ec0), "pH": float(ph0)},
        "train_window": {"start": str(TRAIN_START), "end": str(TRAIN_END)},
    }
    out_path = HOME + "digester_prof_final_v5.json"
    with open(out_path, "w") as f:
        json.dump(out, f, indent=2)
    print(f"\n  Saved: {out_path}")
    print(f"  Total time: {time.time()-t_wall:.0f} s")
    print(f"\n  Tip: re-run to warm-start from v5 for another 25-min pass.")
    print("\n  Done.")


if __name__ == "__main__":
    main()

  DIGESTER v5 — RETRAIN A + B + G ON MARCH 2026 DATA
  Train : 2026-03-15 21:52:00+00:00  ->  2026-03-16 00:09:36+00:00

  Loading data ...
  Loading CSVs:
                     EC_mS: 130135 rows
                        pH: 157425 rows
                    Temp_C:  36857 rows
                    ORP_mV: 126514 rows
                    MFC_mV: 112335 rows
                Q_urine_mL:   3690 rows
              Q_acetate_mL:   3711 rows
           Q_phosphoric_mL:   1334 rows
  Full grid: 242004 pts
  Train: 1630 pts  (2.26 h)
  Nominal: EC0=3.1499  pH0=9.2412  T0=18.75

  STEP 1 — RETRAIN A, B, G, h0, T_ref  (35 parameters)
  Training arrays: 1630 pts  (2.26 h)  buffer 0.1 MB
  Warm start from ./digester_prof_final_v5.json  (T_ref=18.7  h0=[-0.047614  0.05542 ])
  35 parameters (A:16, B:12, G:4, h0:2, T_ref:1)
  Initial cost: 70.8203
  Optimising for 25 min ...
  eval    500  cost=26438.0152  best=70.8134  stall=500  (13s/1500s)
  eval   1000  cost=46489.4214  best=70.8109  stall=1000  (27

## Cell 1 — Imports & Config

In [2]:
import json, time
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.linalg import expm, solve_discrete_are, inv
from scipy.optimize import minimize as sp_minimize
from tqdm.notebook import tqdm

np.set_printoptions(precision=6, suppress=True, linewidth=120)

HOME       = "./"
MODEL_JSON = "digester_prof_final_v2.json"
DT         = 5.0
nx         = 4
PUMP_OFF   = pd.Timestamp("2026-03-09 00:00:00", tz="UTC")  # after val window; dosing active in new data
CHUNK_H    = 12
TIME_LIMIT = 25 * 60

# Training window: Mar 7 02:48 -> Mar 7 14:48 (50:50 urine/acetate)
TRAIN_START = pd.Timestamp("2026-03-07 02:48:00", tz="UTC")
TRAIN_END   = pd.Timestamp("2026-03-07 14:48:00", tz="UTC")
VAL_END     = pd.Timestamp("2026-03-08 02:48:00", tz="UTC")  # 12h validation

BG='#0b1120'; PNL='#111827'; GRD='#1e293b'; TXT='#94a3b8'; TTL='#e2e8f0'; SIM='#22d3ee'
_fc = [0]

def show_fig(fig, name="fig"):
    fn = f"v3_{name}.png"
    fig.savefig(fn, dpi=120, facecolor=fig.get_facecolor(), bbox_inches='tight')
    plt.close(fig)
    print(f"  Saved: {fn}")

def r2(a, p):
    ss = np.sum((a-p)**2); st = np.sum((a-a.mean())**2)
    return 1 - ss/st if st > 0 else float('nan')

def rmse(a, p): return np.sqrt(np.mean((a-p)**2))

def sty(ax, yl):
    ax.set_facecolor(PNL); ax.set_ylabel(yl, color=TXT, fontsize=10)
    ax.tick_params(colors=TXT, labelsize=9)
    for s in ax.spines.values(): s.set_color(GRD)
    ax.grid(True, alpha=0.2, color=GRD, linewidth=0.5)

def discretize(A, B, G, dt):
    A_d = expm(A * dt)
    M = np.zeros((nx+3, nx+3)); M[:nx,:nx]=A*dt; M[:nx,nx:]=B*dt
    B_d = expm(M)[:nx, nx:]
    M2 = np.zeros((nx+1, nx+1)); M2[:nx,:nx]=A*dt; M2[:nx,nx]=G*dt
    G_d = expm(M2)[:nx, nx]
    return A_d, B_d, G_d

def design_observer_dare(A_d, H, Q_obs, R_obs):
    P = solve_discrete_are(A_d.T, H.T, Q_obs, R_obs)
    S = H @ P @ H.T + R_obs
    L = P @ H.T @ inv(S)
    return L, np.linalg.eigvals(A_d - L @ H), P

print("  Imports OK")
print(f"  Train: {TRAIN_START.date()} -> {TRAIN_END.date()}")
print(f"  Valid: {TRAIN_END.date()} -> {VAL_END.date()}")


  Imports OK
  Train: 2026-03-07 -> 2026-03-07
  Valid: 2026-03-07 -> 2026-03-08


## Cell 2 — Load Model & Data

In [2]:
# load fixed B from v2
with open(MODEL_JSON) as f: M_old = json.load(f)
B_fixed = np.array(M_old['B'])
sn = ["EC","pH","h1","h2"]
print("  B (fixed from v2):")
for i in range(4):
    print(f"    {sn[i]:>4s} " + "".join(f"{B_fixed[i,j]:>14.6f}" for j in range(3)))

def load_csv(path):
    df = pd.read_csv(path)
    df.columns = [c.strip().lower() for c in df.columns]
    df["time"] = pd.to_datetime(df["time"], utc=True)
    df = df.sort_values("time").drop_duplicates(subset="time").set_index("time")
    df["value"] = pd.to_numeric(df["value"], errors="coerce")
    return df

def load_all():
    raw = {
        "EC_mS":           load_csv(HOME+"Digester Tank 1-cloud_EC_mS.csv"),
        "pH":              load_csv(HOME+"Digester Tank 1-cloud_pH.csv"),
        "Temp_C":          load_csv(HOME+"Digester Tank 1-cloud_Temp_C.csv"),
        "ORP_mV":          load_csv(HOME+"Digester Tank 1-cloud_ORP_mV.csv"),
        "MFC_mV":          load_csv(HOME+"Digester Tank 1-cloud_MFC_mV.csv"),
        "Q_urine_mL":      load_csv(HOME+"Digester Tank 1-cloud_Q_urine_mL.csv"),
        "Q_acetate_mL":    load_csv(HOME+"Digester Tank 1-cloud_Q_spir_mL.csv"),
        "Q_phosphoric_mL": load_csv(HOME+"Digester Tank 1-cloud_Q_kombucha_mL.csv"),
    }
    for nm, d in raw.items():
        print(f"    {nm:>20s}: {len(d):>6d} rows  [{d.index.min()} -> {d.index.max()}]")
    return raw

def resample_merge(dfs, freq="5s"):
    all_starts  = [df.index.min() for df in dfs.values()]
    sensor_ends = [df.index.max() for n, df in dfs.items() if "Q_" not in n]
    idx = pd.date_range(start=max(all_starts).floor(freq),
                        end=min(sensor_ends).ceil(freq), freq=freq, tz="UTC")
    res = {}; orp_real = None; mfc_real = None
    for name, df in dfs.items():
        if "Q_" in name:
            s = df["value"].resample(freq).sum().reindex(idx).fillna(0.0)
        elif name in ("ORP_mV", "MFC_mV"):
            s = df["value"].resample(freq).mean().reindex(idx)
            real = s.notna()
            if name == "ORP_mV": orp_real = s.index[real]
            else:                mfc_real = s.index[real]
            s = s.ffill()
        else:
            s = df["value"].resample(freq).mean().reindex(idx).interpolate(method="time")
        res[name] = s
    merged = pd.DataFrame(res, index=idx).dropna()
    off = merged.index >= PUMP_OFF
    if off.any():
        merged.loc[off, "Q_urine_mL"]      = 0.0
        merged.loc[off, "Q_acetate_mL"]    = 0.0
        merged.loc[off, "Q_phosphoric_mL"] = 0.0
        print(f"  PUMP SHUTOFF: zeroed {off.sum()} rows after {PUMP_OFF}")
    merged["_ORP_real"] = merged.index.isin(orp_real) if orp_real is not None else True
    merged["_MFC_real"] = merged.index.isin(mfc_real) if mfc_real is not None else True
    return merged

print("\n  Loading data ...")
raw = load_all()
df_full = resample_merge(raw, freq=f"{int(DT)}s")

# slice to training and validation windows
df_train = df_full[(df_full.index >= TRAIN_START) & (df_full.index < TRAIN_END)]
df_val   = df_full[(df_full.index >= TRAIN_END)   & (df_full.index < VAL_END)]

ec0   = float(df_train["EC_mS"].iloc[:720].mean())
ph0   = float(df_train["pH"].iloc[:720].mean())
temp0 = float(df_train["Temp_C"].iloc[:720].mean())

print(f"\n  Train: {df_train.index.min()} -> {df_train.index.max()}")
print(f"  Valid: {df_val.index.min()} -> {df_val.index.max()}")
print(f"  Train pts: {len(df_train)}  |  Val pts: {len(df_val)}")
print(f"  Nominal: EC0={ec0:.4f}  pH0={ph0:.4f}  T0={temp0:.2f}")


  B (fixed from v2):
      EC       0.003243     -0.008060      0.000579
      pH       0.000206      0.000563     -0.000205
      h1       0.000007      0.000012      0.000016
      h2       0.000043      0.000081     -0.000008

  Loading data ...
                   EC_mS: 184108 rows  [2026-02-22 00:00:00.279035822+00:00 -> 2026-03-08 20:54:24.146958022+00:00]
                      pH: 183649 rows  [2026-02-22 00:00:00.279035822+00:00 -> 2026-03-08 20:54:29.274455150+00:00]
                  Temp_C:  43151 rows  [2026-02-22 00:00:25.363279944+00:00 -> 2026-03-08 20:54:14.122797789+00:00]
                  ORP_mV: 137852 rows  [2026-02-22 00:00:00.279035822+00:00 -> 2026-03-08 20:54:24.146958022+00:00]
                  MFC_mV: 126540 rows  [2026-02-22 00:00:05.430483283+00:00 -> 2026-03-08 20:54:24.146958022+00:00]
              Q_urine_mL:   8372 rows  [2026-02-22 00:00:55.412131031+00:00 -> 2026-03-08 20:49:59.184573337+00:00]
            Q_acetate_mL:   8059 rows  [2026-02-22 00:0

## Cell 3 — Build Training Chunks (12h)

In [3]:
chunk_size = int(CHUNK_H * 3600 / DT)
chunks_df  = []
for s in range(0, len(df_train)-chunk_size+1, chunk_size):
    chunks_df.append(df_train.iloc[s:s+chunk_size])
rem = len(df_train) - len(chunks_df)*chunk_size
if rem > chunk_size//2:
    chunks_df.append(df_train.iloc[len(chunks_df)*chunk_size:])

CA = []
for ch in chunks_df:
    CA.append({
        'EC':  ch['EC_mS'].values.astype(np.float64),
        'pH':  ch['pH'].values.astype(np.float64),
        'U':   ch[['Q_urine_mL','Q_acetate_mL','Q_phosphoric_mL']].values.astype(np.float64),
        'T':   ch['Temp_C'].values.astype(np.float64),
        'ORP': ch['ORP_mV'].values.astype(np.float64),
        'MFC': ch['MFC_mV'].values.astype(np.float64),
        'N':   len(ch),
    })
print(f"  {len(CA)} chunks x {CHUNK_H}h ({chunk_size} pts each)")


  1 chunks x 12h (8640 pts each)


## Cell 4 — Optimise A & G (B fixed, 25 min)
Run this cell multiple times if needed — it warm starts each time.

In [3]:
# ── pre-extract training arrays ONCE outside cost function ───────────
_EC  = df_train["EC_mS"].values.copy()
_pH  = df_train["pH"].values.copy()
_U   = df_train[["Q_urine_mL","Q_acetate_mL","Q_phosphoric_mL"]].values.copy()
_T   = df_train["Temp_C"].values.copy()
_N   = len(_EC)

# pre-allocate simulation buffer ONCE — reused every eval
_x_buf = np.zeros((_N, nx), dtype=np.float64)

print(f"  Training arrays pre-extracted: {_N} pts")
print(f"  Simulation buffer: {_x_buf.nbytes / 1e6:.1f} MB allocated once")

def sim_continuous_fast(A, G, h_init, Tr):
    """Reuses pre-allocated buffer — no new memory each call."""
    x0 = np.array([ec0, ph0, h_init[0], h_init[1]])
    _x_buf[0,0] = _EC[0]; _x_buf[0,1] = _pH[0]
    _x_buf[0,2] = h_init[0]; _x_buf[0,3] = h_init[1]
    for k in range(_N-1):
        _x_buf[k+1] = _x_buf[k] + DT*(A@(_x_buf[k]-x0) + B_fixed@_U[k] + G*(_T[k]-Tr))
    ec_err = np.sum((_x_buf[:,0] - _EC)**2)
    ph_err = np.sum((_x_buf[:,1] - _pH)**2)
    return ec_err + ph_err

_ne=[0]; _best=[1e30]; _t=[0.]; _bat=[0]; _bx=[None]
class _Stop(Exception): pass

def cost(theta):
    _ne[0] += 1
    if time.time()-_t[0] > TIME_LIMIT: raise _Stop()
    A_  = theta[:16].reshape(4,4)
    G_  = theta[16:20]
    h0_ = theta[20:22]
    Tr_ = theta[22]
    # cheap stability check before running sim
    evs = np.linalg.eigvals(A_)
    max_ev = max(ev.real for ev in evs)
    if max_ev > 0.01:
        tot = _best[0] + 1e6 * max_ev**2
        return tot
    tot = sim_continuous_fast(A_, G_, h0_, Tr_)
    # soft stability penalty
    for ev in evs:
        if ev.real > 0: tot += 1e6 * ev.real**2
    if tot < _best[0]*0.9995: _best[0]=tot; _bat[0]=_ne[0]; _bx[0]=theta.copy()
    elif tot < _best[0]:      _best[0]=tot; _bx[0]=theta.copy()
    if _ne[0] % 20 == 0:
        print(f"  eval {_ne[0]:>5d}  cost={tot:.2f}  best={_best[0]:.2f}  "
              f"stall={_ne[0]-_bat[0]}  ({time.time()-_t[0]:.0f}s)", flush=True)
    return tot

# warm start from previous run if A exists, else cold start from v2
try:
    theta_warm = np.concatenate([A.ravel(), G, h0, [Tr]])
    print("  Warm start from previous run")
except NameError:
    theta_warm = np.concatenate([np.array(M_old['A']).ravel(),
                                  np.array(M_old['G']),
                                  np.array(M_old['h0']),
                                  [float(M_old['T_ref'])]])
    print("  Cold start from v2")

_t[0]=time.time(); ic=cost(theta_warm)
_ne[0]=0; _best[0]=ic; _bat[0]=0; _bx[0]=theta_warm.copy(); _t[0]=time.time()
print(f"  Initial cost: {ic:.2f}")
print(f"  Optimising continuously for 25 min ...")

try:
    res=sp_minimize(cost, theta_warm, method='Nelder-Mead',
                    options={'maxiter':100000,'xatol':1e-10,'fatol':1e-10,'adaptive':True})
    bx=res.x
    print(f"\n  CONVERGED: cost={res.fun:.4f}  evals={_ne[0]}  ({time.time()-_t[0]:.0f}s)")
except _Stop:
    bx=_bx[0]
    print(f"\n  TIME LIMIT: best={_best[0]:.4f}  evals={_ne[0]}  ({time.time()-_t[0]:.0f}s)")
except KeyboardInterrupt:
    bx=_bx[0]
    print(f"\n  INTERRUPTED: best={_best[0]:.4f}  evals={_ne[0]}")

A=bx[:16].reshape(4,4); G=bx[16:20]; h0=bx[20:22]; Tr=bx[22]; B=B_fixed

# final check using buffer
_x_buf[0,0]=_EC[0]; _x_buf[0,1]=_pH[0]; _x_buf[0,2]=h0[0]; _x_buf[0,3]=h0[1]
x0=np.array([ec0,ph0,h0[0],h0[1]])
for k in range(_N-1):
    _x_buf[k+1] = _x_buf[k] + DT*(A@(_x_buf[k]-x0) + B_fixed@_U[k] + G*(_T[k]-Tr))

print(f"\n  Continuous sim check:")
print(f"    EC  R2={r2(_EC, _x_buf[:,0]):.4f}  RMSE={rmse(_EC, _x_buf[:,0]):.4f}")
print(f"    pH  R2={r2(_pH, _x_buf[:,1]):.4f}  RMSE={rmse(_pH, _x_buf[:,1]):.4f}")
print(f"    EC final={_x_buf[-1,0]:.4f}  pH final={_x_buf[-1,1]:.4f}")

print(f"\n  Eigenvalues of A:")
for i,ev in enumerate(np.linalg.eigvals(A)):
    tc=abs(1/ev.real)/60 if abs(ev.real)>1e-10 else float('inf')
    print(f"    l_{i+1}={ev.real:+.6f} (tau={tc:.1f}min)  stable={ev.real<0}")
print(f"  h0={h0}  T_ref={Tr:.2f}")
print(f"\n  >>> Run cell 4 again for another 25min pass, or proceed to cell 5")

NameError: name 'df_train' is not defined

In [18]:
def sim_chunk(A, G, h_init, cd, Tr):
    """Simulate one chunk using the continuous (Euler) model."""
    N  = cd['N']
    U  = cd['U']
    T  = cd['T']
    x0 = np.array([ec0, ph0, h_init[0], h_init[1]])
    xs = np.zeros((N, nx))
    xs[0] = [cd['EC'][0], cd['pH'][0], h_init[0], h_init[1]]
    for k in range(N - 1):
        xs[k+1] = xs[k] + DT * (A @ (xs[k] - x0) + B_fixed @ U[k] + G * (T[k] - Tr))
    return xs

print('sim_chunk defined')

sim_chunk defined


In [19]:
A_d, B_d, G_d = discretize(A, B, G, DT)

axd=[]; aorp=[]; amfc=[]; aorp_mask=[]; amfc_mask=[]
orp_rm = df_train["_ORP_real"].values
mfc_rm = df_train["_MFC_real"].values

offset = 0
for cd in CA:
    n = cd['N']
    xs = sim_chunk(A, G, h0, cd, Tr)
    xd = xs.copy(); xd[:,0]-=ec0; xd[:,1]-=ph0
    axd.append(xd)
    aorp.append(cd['ORP']); amfc.append(cd['MFC'])
    aorp_mask.append(orp_rm[offset:offset+n])
    amfc_mask.append(mfc_rm[offset:offset+n])
    offset += n

axd=np.vstack(axd); aorp=np.concatenate(aorp); amfc=np.concatenate(amfc)
om=np.concatenate(aorp_mask).astype(bool)
mm=np.concatenate(amfc_mask).astype(bool)

co,_,_,_=np.linalg.lstsq(np.column_stack([axd[om],np.ones(om.sum())]),aorp[om],rcond=None)
cm,_,_,_=np.linalg.lstsq(np.column_stack([axd[mm],np.ones(mm.sum())]),amfc[mm],rcond=None)
C_out=np.array([co[:4],cm[:4]]); biases=np.array([co[-1],cm[-1]])

print(f"  C fit on {om.sum()} ORP pts, {mm.sum()} MFC pts")
print(f"\n  NEW C matrix:")
print("      "+"".join(f"{n:>10s}" for n in sn)+"       bias")
print(f"  ORP "+"".join(f"{C_out[0,j]:>10.4f}" for j in range(4))+f"  {biases[0]:>10.4f}")
print(f"  MFC "+"".join(f"{C_out[1,j]:>10.4f}" for j in range(4))+f"  {biases[1]:>10.4f}")

aec=np.concatenate([cd['EC'] for cd in CA]); apc=np.concatenate([cd['pH'] for cd in CA])
aes=np.concatenate([sim_chunk(A,G,h0,cd,Tr)[:,0] for cd in CA])
aps=np.concatenate([sim_chunk(A,G,h0,cd,Tr)[:,1] for cd in CA])
az=(C_out@axd.T).T+biases

print(f"\n  Training metrics:")
for lbl,obs,pred,mask in [("EC",aec,aes,None),("pH",apc,aps,None),
                           ("ORP",aorp,az[:,0],om),("MFC",amfc,az[:,1],mm)]:
    m=mask if mask is not None else np.ones(len(obs),dtype=bool)
    print(f"    {lbl:>4s}  R2={r2(obs[m],pred[m]):.4f}  RMSE={rmse(obs[m],pred[m]):.4f}")

  C fit on 6475 ORP pts, 5617 MFC pts

  NEW C matrix:
              EC        pH        h1        h2       bias
  ORP    50.0366  286.6129   -0.0742    0.1606    -55.3879
  MFC    10.3072    6.4208   -0.0010    0.0285    200.2937

  Training metrics:
      EC  R2=-0.0885  RMSE=0.0764
      pH  R2=0.0477  RMSE=0.1231
     ORP  R2=0.0173  RMSE=32.6679
     MFC  R2=0.0012  RMSE=18.7219


## Cell 5 — Fit C Matrix

## Cell 6 — Redesign Observer (DARE)

In [21]:
H = np.zeros((2,4)); H[0,0]=1; H[1,1]=1
O = np.vstack([H@np.linalg.matrix_power(A_d,i) for i in range(4)])
print(f"  Observability rank: {np.linalg.matrix_rank(O)}/4")

Q_obs=np.diag([0.1,0.1,0.001,0.001]); R_obs=np.diag([0.04,0.01])
try:
    L,obs_eigs,_=design_observer_dare(A_d,H,Q_obs,R_obs); print("  DARE solved")
except Exception as e:
    from scipy.signal import place_poles as pp
    res2=pp(A_d.T,H.T,[0.95,0.93,0.90,0.88]); L=res2.gain_matrix.T
    obs_eigs=np.linalg.eigvals(A_d-L@H); print(f"  Pole placement fallback: {e}")

print(f"\n  L:"); 
for i in range(4): print(f"    {sn[i]:>4s}  {L[i,0]:>12.6f}  {L[i,1]:>12.6f}")
print(f"\n  Observer poles:")
for i,ev in enumerate(obs_eigs):
    mag=abs(ev); tau=-DT/np.log(mag) if 0<mag<1 else float('inf')
    print(f"    p_{i+1}={ev.real:+.6f} |p|={mag:.6f} tau={tau:.1f}s  stable={mag<1}")


  Observability rank: 4/4
  DARE solved

  L:
      EC      0.722287      0.001179
      pH      0.000295      0.910917
      h1      0.001048     -0.002349
      h2      0.013033     -0.035140

  Observer poles:
    p_1=-0.335629 |p|=0.335629 tau=4.6s  stable=True
    p_2=-0.429146 |p|=0.429146 tau=5.9s  stable=True
    p_3=+0.999863 |p|=0.999863 tau=36528.1s  stable=True
    p_4=+0.999863 |p|=0.999863 tau=36528.1s  stable=True


## Cell 7 — Run Observer on Train & Validation

In [12]:
def run_observer(A_d,B_d,G_d,L,H,h_init,Tr,ec0,ph0,df_data):
    xr=np.array([ec0,ph0,h_init[0],h_init[1]])
    oEC=df_data["EC_mS"].values; opH=df_data["pH"].values
    oTp=df_data["Temp_C"].values
    U=df_data[["Q_urine_mL","Q_acetate_mL","Q_phosphoric_mL"]].values
    N=len(oEC)
    orm=df_data["_ORP_real"].values if "_ORP_real" in df_data.columns else np.ones(N,dtype=bool)
    mfm=df_data["_MFC_real"].values if "_MFC_real" in df_data.columns else np.ones(N,dtype=bool)
    def step(x,temp,u):
        xn=A_d@(x-xr)+xr+B_d@u+G_d*(temp-Tr)
        xn[0]=np.clip(xn[0],0,30); xn[1]=np.clip(xn[1],3,12); return xn
    xp=np.zeros((N,nx)); xc=np.zeros((N,nx)); inn=np.zeros((N,2))
    xc[0]=[oEC[0],opH[0],h_init[0],h_init[1]]; xp[0]=xc[0].copy()
    for k in tqdm(range(1,N), desc="Observer"):
        xpk=step(xc[k-1],oTp[k-1],U[k-1]); xp[k]=xpk
        y=np.array([oEC[k],opH[k]]); innk=y-H@xpk; inn[k]=innk
        xck=xpk+L@innk; xck[0]=np.clip(xck[0],0,30); xck[1]=np.clip(xck[1],3,12); xc[k]=xck
    return xp,xc,inn,orm,mfm

print("  Running observer on training ...")
_,xc_tr,inn_tr,orm_tr,mrm_tr=run_observer(A_d,B_d,G_d,L,H,h0,Tr,ec0,ph0,df_train)
oEC_tr=df_train["EC_mS"].values; opH_tr=df_train["pH"].values
oORP_tr=df_train["ORP_mV"].values; oMFC_tr=df_train["MFC_mV"].values
xd_tr=xc_tr.copy(); xd_tr[:,0]-=ec0; xd_tr[:,1]-=ph0; z_tr=(C_out@xd_tr.T).T+biases

print("\n  TRAINING:")
for lbl,obs,pred,mask in [("EC",oEC_tr,xc_tr[:,0],None),("pH",opH_tr,xc_tr[:,1],None),
                           ("ORP",oORP_tr,z_tr[:,0],orm_tr),("MFC",oMFC_tr,z_tr[:,1],mrm_tr)]:
    m=mask if mask is not None else np.ones(len(obs),dtype=bool)
    print(f"    {lbl:>4s}  R2={r2(obs[m],pred[m]):.4f}  RMSE={rmse(obs[m],pred[m]):.4f}")

h_val=xc_tr[-1,2:]
print(f"\n  Handing off h1={h_val[0]:.6f}  h2={h_val[1]:.6f} to validation")
print("  Running observer on validation ...")
_,xc_v,inn_v,orm_v,mrm_v=run_observer(A_d,B_d,G_d,L,H,h_val,Tr,ec0,ph0,df_val)
oEC_v=df_val["EC_mS"].values; opH_v=df_val["pH"].values
oORP_v=df_val["ORP_mV"].values; oMFC_v=df_val["MFC_mV"].values
xd_v=xc_v.copy(); xd_v[:,0]-=ec0; xd_v[:,1]-=ph0; z_v=(C_out@xd_v.T).T+biases

print("\n  VALIDATION:")
for lbl,obs,pred,mask in [("EC",oEC_v,xc_v[:,0],None),("pH",opH_v,xc_v[:,1],None),
                           ("ORP",oORP_v,z_v[:,0],orm_v),("MFC",oMFC_v,z_v[:,1],mrm_v)]:
    m=mask if mask is not None else np.ones(len(obs),dtype=bool)
    print(f"    {lbl:>4s}  R2={r2(obs[m],pred[m]):.4f}  RMSE={rmse(obs[m],pred[m]):.4f}")


  Running observer on training ...


Observer:   0%|          | 0/8639 [00:00<?, ?it/s]


  TRAINING:
      EC  R2=0.9001  RMSE=0.0231
      pH  R2=0.9837  RMSE=0.0161
     ORP  R2=-0.6321  RMSE=42.0999
     MFC  R2=-0.0190  RMSE=18.9100

  Handing off h1=12.989694  h2=-2.480136 to validation
  Running observer on validation ...


Observer:   0%|          | 0/8639 [00:00<?, ?it/s]


  VALIDATION:
      EC  R2=0.9009  RMSE=0.0244
      pH  R2=0.9868  RMSE=0.0166
     ORP  R2=-3.1709  RMSE=69.7370
     MFC  R2=-0.3936  RMSE=23.1058


## Cell 8 — Plots (saved to PNG)

In [22]:
def plot_4signals(df_data, xc, z, orm, mrm, name, title):
    t=df_data.index
    oEC=df_data["EC_mS"].values; opH=df_data["pH"].values
    oORP=df_data["ORP_mV"].values; oMFC=df_data["MFC_mV"].values
    fig,axes=plt.subplots(4,1,figsize=(16,14),facecolor=BG)
    fig.subplots_adjust(hspace=0.32,left=0.08,right=0.96,top=0.92,bottom=0.05)
    fig.suptitle(title,fontsize=13,color=TTL,fontweight='bold')
    panels=[("EC (mS)",oEC,xc[:,0],None,"#10b981"),("pH",opH,xc[:,1],None,"#f59e0b"),
            ("ORP (mV)",oORP,z[:,0],orm,"#ef4444"),("MFC (mV)",oMFC,z[:,1],mrm,"#a78bfa")]
    for pi,(lbl,obs,pred,mask,clr) in enumerate(panels):
        ax=axes[pi]; sty(ax,lbl)
        ax.plot(t,obs,"-",color=clr,lw=0.8,alpha=0.5,label="Measured")
        ax.plot(t,pred,"-",color=SIM,lw=1.5,alpha=0.9,label="Model")
        m=mask if mask is not None else np.ones(len(obs),dtype=bool)
        ax.text(0.98,0.96,f"R2={r2(obs[m],pred[m]):.4f}  RMSE={rmse(obs[m],pred[m]):.4f}",
                transform=ax.transAxes,ha="right",va="top",fontsize=9,color=SIM,fontfamily="monospace",
                bbox=dict(boxstyle="round,pad=0.3",facecolor=BG,edgecolor=GRD,alpha=0.9))
        leg=ax.legend(fontsize=8,loc="lower right",framealpha=0.8,facecolor=PNL,edgecolor=GRD)
        for lt in leg.get_texts(): lt.set_color(TTL)
    show_fig(fig, name)

plot_4signals(df_train,xc_tr,z_tr,orm_tr,mrm_tr,"training","TRAINING — v4 (Mar 7 02:48–14:48, 50:50 urine/acetate)")
plot_4signals(df_val,xc_v,z_v,orm_v,mrm_v,"validation",f"VALIDATION — v4 (Mar 7 14:48 – Mar 8 02:48)")

# h1/h2 plot
fig,axes=plt.subplots(2,1,figsize=(16,8),facecolor=BG)
fig.subplots_adjust(hspace=0.35,left=0.08,right=0.96,top=0.90,bottom=0.08)
fig.suptitle("Hidden States h1 & h2 — Training Window (50:50 urine/acetate)",fontsize=13,color=TTL,fontweight='bold')
for ri,(si,lbl,clr) in enumerate([(2,"h1","#ef4444"),(3,"h2","#a78bfa")]):
    ax=axes[ri]; sty(ax,lbl); ax.plot(df_train.index,xc_tr[:,si],color=clr,lw=1.0,alpha=0.8)
show_fig(fig, "hidden_states")


  Saved: v3_training.png
  Saved: v3_validation.png
  Saved: v3_hidden_states.png


## Cell 9 — Save digester_prof_final_v4.json

In [14]:
val_m={
    "r2_EC":r2(oEC_v,xc_v[:,0]),"rmse_EC":rmse(oEC_v,xc_v[:,0]),
    "r2_pH":r2(opH_v,xc_v[:,1]),"rmse_pH":rmse(opH_v,xc_v[:,1]),
    "r2_ORP":r2(oORP_v[orm_v],z_v[orm_v,0]),"rmse_ORP":rmse(oORP_v[orm_v],z_v[orm_v,0]),
    "r2_MFC":r2(oMFC_v[mrm_v],z_v[mrm_v,1]),"rmse_MFC":rmse(oMFC_v[mrm_v],z_v[mrm_v,1]),
}
train_m={
    "r2_EC":r2(aec,aes),"rmse_EC":rmse(aec,aes),
    "r2_pH":r2(apc,aps),"rmse_pH":rmse(apc,aps),
    "r2_ORP":r2(aorp[om],az[om,0]),"rmse_ORP":rmse(aorp[om],az[om,0]),
    "r2_MFC":r2(amfc[mm],az[mm,1]),"rmse_MFC":rmse(amfc[mm],az[mm,1]),
}

res_json={
    "version":"v4 — A,G,C retrained Mar7 12h 50:50 urine/acetate; B fixed from v2",
    "dt":DT,"A":A.tolist(),"B":B.tolist(),"G":G.tolist(),
    "A_d":A_d.tolist(),"B_d":B_d.tolist(),"G_d":G_d.tolist(),
    "C":C_out.tolist(),"biases":biases.tolist(),
    "L":L.tolist(),"H":H.tolist(),
    "h0":h0.tolist(),"T_ref":float(Tr),
    "nominal":{"EC_mS":float(ec0),"pH":float(ph0)},
    "train_metrics":train_m,"validation_metrics":val_m,
    "train_window":{"start":str(TRAIN_START),"end":str(TRAIN_END)},
    "val_window":{"start":str(TRAIN_END),"end":str(VAL_END)},
    "pump_off":str(PUMP_OFF),
    "B_source":"fixed from digester_prof_final_v2.json",
}

with open("digester_prof_final_v4.json","w") as f: json.dump(res_json,f,indent=2)
print("  Saved: digester_prof_final_v4.json")
print("\n  VALIDATION METRICS:")
for k,v in val_m.items(): print(f"    {k}: {v:.4f}")


  Saved: digester_prof_final_v4.json

  VALIDATION METRICS:
    r2_EC: 0.9009
    rmse_EC: 0.0244
    r2_pH: 0.9868
    rmse_pH: 0.0166
    r2_ORP: -3.1709
    rmse_ORP: 69.7370
    r2_MFC: -0.3936
    rmse_MFC: 23.1058


In [15]:
# test continuous simulation over full training window
x_cont = np.zeros((len(df_train), nx))
x_cont[0] = [df_train["EC_mS"].iloc[0], df_train["pH"].iloc[0], h0[0], h0[1]]
x0 = np.array([ec0, ph0, h0[0], h0[1]])
U  = df_train[["Q_urine_mL","Q_acetate_mL","Q_phosphoric_mL"]].values
T  = df_train["Temp_C"].values
for k in range(len(df_train)-1):
    x_cont[k+1] = x_cont[k] + DT*(A@(x_cont[k]-x0) + B_fixed@U[k] + G*(T[k]-Tr))

print(f"EC  R2={r2(df_train['EC_mS'].values, x_cont[:,0]):.4f}  RMSE={rmse(df_train['EC_mS'].values, x_cont[:,0]):.4f}")
print(f"pH  R2={r2(df_train['pH'].values,    x_cont[:,1]):.4f}  RMSE={rmse(df_train['pH'].values,    x_cont[:,1]):.4f}")
print(f"EC final: {x_cont[-1,0]:.4f}  pH final: {x_cont[-1,1]:.4f}")

EC  R2=-0.1017  RMSE=0.0769
pH  R2=0.0214  RMSE=0.1248
EC final: 5.3883  pH final: 8.5853


In [23]:
# CONVERGENCE SIMULATION — initialize observer with wrong h1/h2
# Shows observer converges to true states from bad initial conditions

N_conv = min(len(df_train), 10000)  # first ~14 hours
df_conv = df_train.iloc[:N_conv]

# wrong initial h1/h2 — 100x the true values
h_wrong = np.array([h0[0]*100, h0[1]*100])
h_true  = h0.copy()

print(f"  True h0:  h1={h_true[0]:.6f}  h2={h_true[1]:.6f}")
print(f"  Wrong h0: h1={h_wrong[0]:.6f}  h2={h_wrong[1]:.6f}")

_, xc_wrong, _, _, _ = run_observer(A_d,B_d,G_d,L,H,h_wrong,Tr,ec0,ph0,df_conv)
_, xc_true,  _, _, _ = run_observer(A_d,B_d,G_d,L,H,h_true, Tr,ec0,ph0,df_conv)

settle = int(0.25 * N_conv)  # settled = last 75%
h1_rmse = rmse(xc_true[settle:,2], xc_wrong[settle:,2])
h2_rmse = rmse(xc_true[settle:,3], xc_wrong[settle:,3])

print(f"\n  Settled RMSE (last 75%):")
print(f"    h1 RMSE = {h1_rmse:.6f}")
print(f"    h2 RMSE = {h2_rmse:.6f}")

# plot convergence
fig, axes = plt.subplots(2,1,figsize=(14,8),facecolor=BG)
fig.subplots_adjust(hspace=0.35,left=0.08,right=0.96,top=0.90,bottom=0.08)
fig.suptitle("Observer Convergence — Wrong vs True h1/h2 Initial Conditions (v4)",
             fontsize=13,color=TTL,fontweight='bold')
t_conv = df_conv.index
for ri,(si,lbl,clr) in enumerate([(2,'h1','#ef4444'),(3,'h2','#a78bfa')]):
    ax=axes[ri]; sty(ax,lbl)
    ax.plot(t_conv,xc_true[:,si], color=clr,  lw=1.5, alpha=0.9, label='True init')
    ax.plot(t_conv,xc_wrong[:,si],color='white',lw=1.0,alpha=0.6,ls='--',label='Wrong init')
    leg=ax.legend(fontsize=8,framealpha=0.8,facecolor=PNL,edgecolor=GRD)
    for lt in leg.get_texts(): lt.set_color(TTL)
show_fig(fig,"convergence")

  True h0:  h1=0.007082  h2=-0.000200
  Wrong h0: h1=0.708222  h2=-0.019992


Observer:   0%|          | 0/8639 [00:00<?, ?it/s]

Observer:   0%|          | 0/8639 [00:00<?, ?it/s]


  Settled RMSE (last 75%):
    h1 RMSE = 0.701140
    h2 RMSE = 0.019792
  Saved: v3_convergence.png


In [1]:
# ── PLOT EC & pH: Training + Validation (add at end, no re-running needed) ──

import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(2, 2, figsize=(18, 8), facecolor=BG)
fig.subplots_adjust(hspace=0.35, wspace=0.25, left=0.07, right=0.97, top=0.90, bottom=0.08)
fig.suptitle("EC & pH — Observer Fit (v4, Mar 7)", fontsize=13, color=TTL, fontweight='bold')

panels = [
    (axes[0,0], df_train.index, df_train["EC_mS"].values, xc_tr[:,0], "EC (mS)",  "#10b981", "TRAINING"),
    (axes[1,0], df_train.index, df_train["pH"].values,    xc_tr[:,1], "pH",       "#f59e0b", "TRAINING"),
    (axes[0,1], df_val.index,   df_val["EC_mS"].values,   xc_v[:,0],  "EC (mS)",  "#10b981", "VALIDATION"),
    (axes[1,1], df_val.index,   df_val["pH"].values,      xc_v[:,1],  "pH",       "#f59e0b", "VALIDATION"),
]

for ax, t, obs, pred, ylabel, clr, split in panels:
    sty(ax, ylabel)
    ax.plot(t, obs,  "-", color=clr,   lw=0.9, alpha=0.55, label="Measured")
    ax.plot(t, pred, "-", color=SIM,   lw=1.6, alpha=0.95, label="Model")
    r2_val   = r2(obs, pred)
    rmse_val = rmse(obs, pred)
    ax.set_title(split, color=TTL, fontsize=10, fontweight='bold')
    ax.text(0.98, 0.96,
            f"R²={r2_val:.4f}   RMSE={rmse_val:.4f}",
            transform=ax.transAxes, ha="right", va="top",
            fontsize=9, color=SIM, fontfamily="monospace",
            bbox=dict(boxstyle="round,pad=0.3", facecolor=BG, edgecolor=GRD, alpha=0.9))
    leg = ax.legend(fontsize=8, loc="lower right", framealpha=0.8, facecolor=PNL, edgecolor=GRD)
    for lt in leg.get_texts(): lt.set_color(TTL)
    ax.xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter("%m/%d %H:%M"))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=20, ha="right", fontsize=7)

show_fig(fig, "ec_ph_train_val")
print("  Saved: v3_ec_ph_train_val.png")

NameError: name 'BG' is not defined